# Syntropy2 — Enterprise Benchmark & Verdict

**This notebook exists to answer one simple question:**

> ### Is this technique — well calibrated and correctly implemented — actually *superior* to the old traditional compression methods? By **how much**, in **quality**, **speed**, **size** and **computational cost**? And is the **1.245 bpc goal** we set earlier still the optimal target?

**The subject.** A ≈10.6 M-parameter *neuro-symbolic* compressor: a windowed
Transformer + a GRU residual expert + classical order-0/1/2 and repeat-match
statistics, blended by a small learned gate. The blend predicts a probability
for every next token, and an **arithmetic range coder** turns those
probabilities into bits. That construction is lossless by definition — the
model never guesses bytes, it only prices them: a token the model considers
likely costs a fraction of a bit, a surprising one costs many.

**The yardstick.** `bpc` = *bits per character* (identical to bits/byte on
enwik8, which is 100 MB of English Wikipedia XML, 1 byte per character).
Uncompressed text is 8.0 bpc; `gzip -9` reaches ≈2.8 on our test split;
**lower is better**, and compressed size = `bpc / 8 ×` original size.
All quality numbers are measured on a **held-out 5 MB test split** the model
never saw during training.

**Layout — run top to bottom.**

| Part | Suites | What happens | Cost |
|---|---|---|---|
| **1 — Evidence** | A–G | load the exported checkpoint; measure quality, prove losslessness, calibrate, measure speed/robustness, apply the stopping policy | one heavy **RUN CELL** (minutes on GPU, ~1 h capped on CPU) |
| **2 — Verdict** | H–K | measure gzip/bzip2/xz/zstd/brotli/PPMd on the *same bytes*, draw the comparison dashboard, re-audit the 1.245 goal against the *full* training history, print the answer | ~2–5 min; **works without re-running Part 1** (it reuses the newest saved `benchmark_report.json`) |

**On Kaggle:** attach the dataset `christianocanab/syntropy2-ckpt` (checkpoints,
enwik8, `neuro_symbolic_compressor.py`). A T4 GPU accelerates Part 1; Part 2 is
CPU-bound either way.


### Where each part of the question gets answered

| Sub-question | Evidence | Answer artifact |
|---|---|---|
| Is it *correct* (truly lossless)? | Suite **B** hard gates | PASS/FAIL, non-zero exit on failure |
| Is it *well calibrated*? | Suite **C** temperature search | `T*`, ECE, calibrated bpc |
| Quality — how many bits? | Suites **A** + **H** | chart *Quality*, `benchmark_report.json` |
| Speed — how fast, both directions? | Suites **D** + **H** | chart *Speed*, *Trade-off* |
| Size — who ships fewer bytes end-to-end? | Suite **H** footprint | chart *Break-even* |
| Computational cost? | Suite **H** | chart *Cost* |
| Keep training this checkpoint? | Suite **F** | CONTINUE / STOP verdict |
| **Is 1.245 bpc still the optimal goal?** | Suite **J** | chart *Goal audit*, `goal_audit.json` |
| **The one-line answer** | Suite **K** | KPI tiles + `verdict.md` |


---
## Part 1 — Evidence: the standalone benchmark harness

The next cell is the harness header: imports plus the module docstring listing
suites A–F. Everything in Part 1 is **training-free** — it evaluates an
*exported* checkpoint (`best.pt` + `tokenizer.json`), so results are
reproducible from the published artifacts alone. The same code ships as
`benchmark_enterprise.py` and can run as a CI job (non-zero exit if any
correctness gate fails).


In [ ]:
# =============================================================================
#  SECTION 0 -- BOOTSTRAP (Kaggle / local)
#  Makes neuro_symbolic_compressor.py importable. On Kaggle, attach it as a
#  Dataset or Utility Script; locally, keep it next to this notebook.
# =============================================================================
import os, sys, glob, shutil

try:
    import neuro_symbolic_compressor  # noqa: F401
except ImportError:
    _found = None
    for _pat in ("/kaggle/input/datasets/christianocanab/syntropy2-ckpt/**/neuro_symbolic_compressor.py",
                 "/kaggle/input/**/neuro_symbolic_compressor.py",
                 "/kaggle/usr/lib/**/neuro_symbolic_compressor.py",
                 os.path.join(os.getcwd(), "..", "neuro_symbolic_compressor.py")):
        _hits = sorted(glob.glob(_pat, recursive=True))
        if _hits:
            _found = _hits[0]
            break
    if _found is None:
        raise ImportError(
            "neuro_symbolic_compressor.py not found. On Kaggle: Add Input -> "
            "attach the dataset/utility script that contains it. Locally: place "
            "it in the same folder as this notebook.")
    shutil.copy(_found, os.path.join(os.getcwd(), "neuro_symbolic_compressor.py"))
    import neuro_symbolic_compressor  # noqa: F401

print("[bootstrap] neuro_symbolic_compressor:", neuro_symbolic_compressor.__file__)


In [ ]:
# -*- coding: utf-8 -*-
# =============================================================================
#  ENTERPRISE BENCHMARK & STOPPING-POLICY HARNESS  (standalone, training-free)
# =============================================================================
#  Professional evaluation of an exported neuro-symbolic compressor checkpoint
#  (results/export/best.pt + tokenizer.json) WITHOUT re-running training.
#
#  Test suites (industry-standard compressor + ML metrics):
#    A. Compression quality   : bits/byte (raw + temperature-calibrated),
#                               perplexity, top-1/top-5 accuracy, projected
#                               compressed size, ratio vs gzip/bzip2/xz(/zstd).
#    B. Correctness gates     : arithmetic-coder self-test, byte-exact
#                               encode->decode roundtrip on REAL model
#                               distributions, tokenizer roundtrip, streaming
#                               reference-decoder parity.  Hard pass/fail.
#    C. Calibration           : optimal temperature T*, NLL before/after, ECE.
#    D. Performance           : throughput (tok/s, MB/s), latency p50/p95,
#                               parameter count, artifact sizes, peak memory.
#    E. Robustness (stress)   : repetitive / random / structured-text inputs.
#    F. Stopping policy       : overtraining check (latest vs best checkpoint),
#                               marginal-gain extrapolation to the end of the
#                               curriculum, and a CONTINUE / STOP verdict.
#
#  Usage:
#      python benchmark_enterprise.py                  # evaluate results/export
#      python benchmark_enterprise.py --full           # no batch caps (slow)
#      python benchmark_enterprise.py --train --budget 80   # train first, then bench
#
#  SKIP_TRAINING=True (default) evaluates the exported previous-session result
#  found under results/.  Set it to False (or pass --train) to run a training
#  session first via run_budget() and benchmark its freshest checkpoint.
#
#  Exit code is non-zero if any hard correctness gate fails, so this file can
#  run as a CI job.
# =============================================================================
from __future__ import annotations

import os
import sys
import json
import time
import math
import gzip
import bz2
import lzma
import argparse
import platform
import resource
from dataclasses import asdict

import numpy as np
import torch

import neuro_symbolic_compressor as nsc
from neuro_symbolic_compressor import (
    Config, HybridLM, make_loader, move_batch, count_params,
    _xla_safe_load, _selftest_coder, _quantize, _RangeCoder,
    encode_with, decode_with, reference_decoder_check,
)

### Configuration — the knobs that matter

* **`SKIP_TRAINING = True`** — benchmark the exported results only. (`--train`
  would run a training session first; not needed here.)
* **The goal block** — `GOAL_TEST_BPB = 1.245` is the stopping target set on
  2026-07-06 by extrapolating the *first two* checkpoints on record
  (val 1.3100 @ 67.3k → 1.3071 @ 72k ⇒ projected calibrated test
  1.2437 ± 0.0035). It is a *projection*, not a law — **Suite J re-audits it
  against the full history**, which is exactly what the final part of our
  question demands.
* **Eval budget** — by default the test split is evaluated on a capped subset
  (~196k tokens ≈ 12% of the split) so a laptop CPU can finish. Capped numbers
  are honest *estimates with slice noise*: the same checkpoint measured
  1.2672 bpc on the capped slice vs 1.2860 bpc on the full split. Pass
  `--full` (GPU recommended) for confirmatory numbers; the notebook labels
  which kind it is using.


In [ ]:
# =============================================================================
#  CONFIG — edit here (every value can also be overridden on the CLI)
# =============================================================================
SKIP_TRAINING = True          # True: benchmark results/export only. False: train first.

# ---- stopping policy (Suite F): when is the model "done"? ------------------
# GOAL_TEST_BPB is the statistically-projected BEST achievable calibrated test
# bits/byte (== bits/char on enwik8) for this architecture at curriculum end
# (step 160k).  Ensemble of decay models (linear-rate-decay, power laws
# alpha=0.2..1.0, cosine-LR-integral) fitted to the recorded checkpoints
# (val 1.3100@67.3k -> 1.3071@72k, cal-test 1.2693@72k, 2026-07-06) projects a
# final calibrated test bpb of 1.2437 +/- 0.0035; 1.245 is the ensemble median.
# Training continues across sessions until this value is CONFIRMED, then
# run_budget() auto-stops (keep in sync with
# neuro_symbolic_compressor.GOAL_TEST_BPB).
GOAL_TEST_BPB   = 1.245   # absolute goal: stop sessions once calibrated test bpb <= this
PLATEAU_ABS_BPB = 0.003   # stop when a session (~30k steps) gains less val bpb than this...
PLATEAU_REL     = 0.0025  # ...or less than 0.25% relative (whichever is larger)
OVERFIT_TOL     = 0.005   # latest-vs-best val bpb regression (abs) that flags overtraining

# ---- evaluation budget (CPU-friendly defaults; --full removes the caps) ----
EVAL_BATCH_SIZE   = 4     # sequences per eval batch (smaller = less RAM)
TEST_MAX_BATCHES  = 48    # test-split batches  (48 * 4 * 1024 ≈ 196k tokens ≈ 0.6 MB)
VAL_MAX_BATCHES   = 24    # val-split batches for calibration + overtraining check
CALIB_BATCHES     = 8     # batches whose logits feed temperature search / ECE
CALIB_POS_PER_SEQ = 128   # random positions sampled per sequence (bounds logit RAM)
STRESS_MAX_BATCHES = 4    # batches per stress case
REF_DECODER_N     = 96    # tokens for the sequential reference-decoder parity check

# ---- training mode (only used when SKIP_TRAINING=False / --train) ----------
TRAIN_BUDGET_H  = None    # total cross-session budget (None -> run_budget prompts/loads)
SESSION_CAP_H   = 11.5

# Notebook-safe: __file__ is undefined when these sections run as ipynb cells.
BASE_DIR = (os.path.dirname(os.path.abspath(__file__))
            if "__file__" in globals() else os.getcwd())

### Artifact & data discovery

Resolves where the exported checkpoint and enwik8 live, in priority order:
`results/` (a previous session's outputs) → local `export/` and `data/` →
any attached Kaggle input dataset. Downloads enwik8 only if truly absent.
Everything later in the notebook goes through these paths, so local and
Kaggle runs behave identically.


In [ ]:
# =============================================================================
#  Artifact / data discovery
# =============================================================================
def locate_paths(export_dir=None, data_path=None):
    """Resolve exported-checkpoint dir + enwik8 path. Prefers results/ (previous
    session results), then the live export/ and data/ dirs, then any attached
    Kaggle input dataset."""
    import glob as _glob
    exp_candidates = ([export_dir] if export_dir else []) + [
        os.path.join(BASE_DIR, "results", "export"),
        os.path.join(BASE_DIR, "export"),
    ] + sorted(os.path.dirname(p) for p in
               _glob.glob("/kaggle/input/**/tokenizer.json", recursive=True))
    exp = next((d for d in exp_candidates
                if d and os.path.exists(os.path.join(d, "tokenizer.json"))), None)
    if exp is None:
        raise FileNotFoundError(f"no export dir with tokenizer.json in {exp_candidates}")

    data_candidates = ([data_path] if data_path else []) + [
        os.path.join(BASE_DIR, "results", "data", "enwik8"),
        os.path.join(BASE_DIR, "data", "enwik8"),
    ] + sorted(_glob.glob("/kaggle/input/**/enwik8", recursive=True))
    data = next((p for p in data_candidates
                 if p and os.path.exists(p) and os.path.getsize(p) > 90_000_000), None)
    if data is None:
        # SKIP_TRAINING runs never reach the trainer's downloader, so fetch here.
        print(f"[data] enwik8 not found in {data_candidates} — downloading...")
        data = nsc.download_enwik8(writable_dir(os.path.join(BASE_DIR, "data")))

    best = os.path.join(exp, "best.pt")
    if not os.path.exists(best):
        best = nsc._find_pt_file(exp)
    lasts = sorted(
        (p for p in (os.path.join(exp, f) for f in os.listdir(exp))
         if p.endswith(".last")), key=os.path.getmtime)
    return dict(export_dir=exp, data=data, best_ckpt=best,
                last_ckpt=lasts[-1] if lasts else None,
                tokenizer=os.path.join(exp, "tokenizer.json"))


def writable_dir(preferred: str) -> str:
    """Return `preferred` if it can be written to, else ./benchmark_out.
    Kaggle input datasets are read-only, so reports/caches fall back to the
    working directory there."""
    try:
        os.makedirs(preferred, exist_ok=True)
        if os.access(preferred, os.W_OK):
            return preferred
    except OSError:
        pass
    fb = os.path.join(os.getcwd(), "benchmark_out")
    os.makedirs(fb, exist_ok=True)
    return fb


def load_checkpoint_meta(path):
    """Load a checkpoint, returning (state_dict, meta). Handles both the
    weights-only best.pt format and the full-resume .last format."""
    try:
        ck = _xla_safe_load(path, map_location="cpu", weights_only=True)
    except Exception:
        ck = _xla_safe_load(path, map_location="cpu", weights_only=False)
    meta = {"path": path, "size_mb": os.path.getsize(path) / 2**20,
            "step": int(ck.get("step", -1)),
            "val_bpb": float(ck.get("val_bpb", ck.get("best", float("nan")))),
            "stage": ck.get("stage"), "patience": ck.get("patience"),
            "reason": ck.get("reason")}
    return ck["model"], ck["cfg"], meta


class _UniformClassical:
    """Placeholder for HybridLM construction; the real order0/order1 tables are
    buffers inside the checkpoint state_dict and overwrite these on load."""

    def __init__(self, V):
        self.order0 = np.full(V, 1.0 / V, dtype=np.float32)
        self.order1 = np.full((V, V), 1.0 / V, dtype=np.float32)


def build_model(state_dict, cfg_dict):
    cfg = Config(**cfg_dict)
    model = HybridLM(cfg, _UniformClassical(cfg.vocab_size))
    model.load_state_dict(state_dict)
    model.eval()
    return model, cfg

### Rebuilding the classical features (fast, exact)

During training the model was fed order-2 and repeat-match statistics
(`ClassicalStats`). Rebuilding that dict-of-dicts here would need several GB
of RAM, so this cell re-implements it as vectorized NumPy with the same
semantics — order-2 lookups are exact, and the match model uses exact K-gram
keys (strictly *fewer* hash collisions than training, effect ≈ 0). This is
infrastructure, not measurement: it only prepares model inputs.


In [ ]:
# =============================================================================
#  Fast classical features (vectorized rebuild of ClassicalStats.features)
# =============================================================================
#  The training loop fed the model order-2 and match-model (top, conf) features
#  computed by ClassicalStats.  Rebuilding that dict-of-dicts on this machine
#  would need several GB of RAM, so this is a numpy re-implementation with the
#  same semantics: order-2 lookups are exact (tie-break may differ on equal
#  counts); the match model uses exact K-gram integer keys instead of a 31-bit
#  truncated hash (strictly fewer collisions than training, effect ~0).
# =============================================================================
class FastO2:
    def __init__(self, V):
        self.V = V
        self.ctx = self.top = self.conf = None

    def fit(self, tokens: np.ndarray):
        V = self.V
        t = tokens.astype(np.int64)
        codes = (t[:-2] * V + t[1:-1]) * V + t[2:]      # (ctx a*V+b) * V + next
        uniq, counts = np.unique(codes, return_counts=True)
        ctx, nxt = uniq // V, uniq % V
        starts = np.flatnonzero(np.r_[True, ctx[1:] != ctx[:-1]])
        totals = np.add.reduceat(counts, starts)
        order = np.lexsort((nxt, -counts, ctx))          # ctx asc, count desc
        ctx_s, nxt_s, cnt_s = ctx[order], nxt[order], counts[order]
        s2 = np.flatnonzero(np.r_[True, ctx_s[1:] != ctx_s[:-1]])
        self.ctx = ctx_s[s2]
        self.top = nxt_s[s2].astype(np.int64)
        self.conf = (cnt_s[s2] / totals).astype(np.float32)
        return self

    def save(self, path):
        np.savez_compressed(path, ctx=self.ctx, top=self.top, conf=self.conf)

    @classmethod
    def load(cls, path, V):
        z = np.load(path)
        o = cls(V)
        o.ctx, o.top, o.conf = z["ctx"], z["top"], z["conf"]
        return o

    def lookup(self, keys: np.ndarray):
        """Vectorized (top, conf) for an array of a*V+b context keys."""
        idx = np.searchsorted(self.ctx, keys)
        idx = np.clip(idx, 0, len(self.ctx) - 1)
        hit = self.ctx[idx] == keys
        top = np.where(hit, self.top[idx], 0)
        conf = np.where(hit, self.conf[idx], 0.0).astype(np.float32)
        return top, conf


def compute_features(tokens: np.ndarray, o2: FastO2, K: int, cap: float):
    """Aligned per-position features, mirroring ClassicalStats.features()."""
    V = o2.V
    t = tokens.astype(np.int64)
    n = len(t)
    o2_top = np.zeros(n, dtype=np.int64)
    o2_conf = np.zeros(n, dtype=np.float32)
    if n >= 2:  # order-2: context (t[i-1], t[i]) predicts i+1, defined for i>=1
        keys = t[:-1] * V + t[1:]
        top, conf = o2.lookup(keys)
        o2_top[1:], o2_conf[1:] = top, conf

    m_top = np.zeros(n, dtype=np.int64)
    m_conf = np.zeros(n, dtype=np.float32)
    if n > K:  # match model: exact K-gram key ending at i, defined for i>=K
        kg = np.zeros(n - K + 1, dtype=np.int64)   # packed exact K-gram keys
        for j in range(K):
            kg = kg * V + t[j:n - K + 1 + j]
        pos: dict = {}
        run = 0
        for i in range(K, n):
            h = int(kg[i - K + 1])
            prev = pos.get(h)
            if prev is not None and prev + 1 < n:
                m_top[i] = int(t[prev + 1])
                run = run + 1 if m_top[i - 1] == t[i] else 1
                m_conf[i] = min(run / cap, 0.99)
            else:
                run = 0
            pos[h] = i
    return dict(order2_top=o2_top, order2_conf=o2_conf,
                match_top=m_top, match_conf=m_conf)


def encode_bytes_chunked(tok, data: bytes, chunk: int = 1 << 22) -> np.ndarray:
    """Chunked BPE encode (bounded RAM). Chunks split on newline boundaries so
    byte-level BPE tokenization matches the unchunked result."""
    if len(data) <= chunk:
        return encode_with(tok, data)
    parts, i = [], 0
    while i < len(data):
        j = min(len(data), i + chunk)
        if j < len(data):
            k = data.rfind(b"\n", i, j)
            if k > i:
                j = k + 1
        parts.append(encode_with(tok, data[i:j]))
        i = j
    return np.concatenate(parts)


def prepare_eval_data(tok, cfg: Config, data_path: str, cache_dir: str):
    """Split enwik8 exactly like training (train/val/test fractions), rebuild
    the order-2 table from the TRAIN split, and compute cached features for the
    val/test splits only (train features are not needed for evaluation)."""
    os.makedirs(cache_dir, exist_ok=True)
    with open(data_path, "rb") as f:
        data = f.read()
    n = len(data)
    n_tr, n_va = int(n * cfg.train_frac), int(n * cfg.val_frac)
    byts = {"train": data[:n_tr], "val": data[n_tr:n_tr + n_va],
            "test": data[n_tr + n_va:]}
    del data

    key = f"v{cfg.vocab_size}_n{n}"
    toks = {}
    for split in ("train", "val", "test"):
        p = os.path.join(cache_dir, f"tokens_{split}_{key}.npy")
        if os.path.exists(p):
            toks[split] = np.load(p)
            print(f"  [cache] tokens {split}: HIT ({len(toks[split]):,})")
        else:
            t0 = time.time()
            toks[split] = encode_bytes_chunked(tok, byts[split])
            np.save(p, toks[split])
            print(f"  [cache] tokens {split}: encoded {len(toks[split]):,} "
                  f"({time.time()-t0:.0f}s)")

    o2_path = os.path.join(cache_dir, f"o2_{key}.npz")
    if os.path.exists(o2_path):
        o2 = FastO2.load(o2_path, cfg.vocab_size)
        print(f"  [cache] order-2 table: HIT ({len(o2.ctx):,} contexts)")
    else:
        t0 = time.time()
        o2 = FastO2(cfg.vocab_size).fit(toks["train"])
        o2.save(o2_path)
        print(f"  [cache] order-2 table: fit on train "
              f"({len(o2.ctx):,} contexts, {time.time()-t0:.0f}s)")

    feats = {}
    for split in ("val", "test"):
        p = os.path.join(cache_dir, f"feats_{split}_{key}.npz")
        if os.path.exists(p):
            z = np.load(p)
            feats[split] = {k: z[k] for k in z.files}
            print(f"  [cache] features {split}: HIT")
        else:
            t0 = time.time()
            feats[split] = compute_features(toks[split], o2,
                                            cfg.order2_match_order,
                                            float(cfg.match_len_cap))
            np.savez_compressed(p, **feats[split])
            print(f"  [cache] features {split}: computed ({time.time()-t0:.0f}s)")
    return byts, toks, feats, o2


def eval_loader(tokens, feats, cfg: Config, batch_size: int):
    ecfg = Config(**{**asdict(cfg), "batch_size": batch_size})
    return make_loader(tokens, feats, ecfg, shuffle=False)

### Suite A — Compression quality (plus Suite D's latency probes)

The core measurement. For every position the model outputs a probability for
the true next token; an ideal arithmetic coder pays exactly `−log₂ p` bits for
it. Cross-entropy summed over the split ÷ bytes = **bits/byte = bpc**. No
compressed file needs to be written to know the size — Suite B proves the
coder achieves this bound to within mathematical rounding.
Also collected here: perplexity, top-1/top-5 next-token accuracy (intuition:
how often the model's first guess is right), batch latency percentiles and
throughput. `baseline_bpb()` runs gzip/bzip2/xz (+ zstd/brotli when installed)
on the **same bytes** for the ratio comparison.


In [ ]:
# =============================================================================
#  Suite A + D core: model evaluation with quality/latency metrics
# =============================================================================
@torch.no_grad()
def eval_metrics(model, loader, tokens_total, bytes_total,
                 max_batches=None, temperature=1.0, device=None):
    """Cross-entropy -> bits/byte plus top-k accuracy and latency percentiles."""
    device = device or nsc.DEVICE
    model.eval()
    nats, ntok, top1, top5 = 0.0, 0, 0, 0
    lat = []
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        batch = move_batch(batch, device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        out = model(batch, stage=2)
        if device.type == "cuda":
            torch.cuda.synchronize()
        lat.append(time.time() - t0)
        z = out["z_final"] / temperature
        y = batch["target_ids"]
        ce = torch.nn.functional.cross_entropy(
            z.reshape(-1, z.shape[-1]), y.reshape(-1), reduction="sum")
        nats += ce.item()
        ntok += y.numel()
        tk = z.topk(5, dim=-1).indices
        top1 += (tk[..., 0] == y).sum().item()
        top5 += (tk == y.unsqueeze(-1)).any(-1).sum().item()
    npt = nats / max(1, ntok)
    ratio = tokens_total / max(1, bytes_total)
    lat_s = sorted(lat)
    seq_tokens = ntok // max(1, len(lat))
    return dict(
        nats_per_token=npt,
        perplexity=math.exp(min(50.0, npt)),
        bits_per_byte=npt * math.log2(math.e) * ratio,
        top1_acc=top1 / max(1, ntok),
        top5_acc=top5 / max(1, ntok),
        tokens_evaluated=ntok,
        coverage_frac=ntok / max(1, tokens_total),
        latency_ms_p50=1e3 * lat_s[len(lat_s) // 2] if lat_s else float("nan"),
        latency_ms_p95=1e3 * lat_s[int(len(lat_s) * 0.95)] if lat_s else float("nan"),
        throughput_tok_s=ntok / max(1e-9, sum(lat)),
        batch_tokens=seq_tokens,
    )


def baseline_bpb(data: bytes):
    """Classical-compressor baselines (stdlib + optional zstd/brotli)."""
    out = {
        "gzip-9": len(gzip.compress(data, 9)) * 8.0 / len(data),
        "bzip2-9": len(bz2.compress(data, 9)) * 8.0 / len(data),
        "xz-9": len(lzma.compress(data, preset=9)) * 8.0 / len(data),
    }
    try:
        import zstandard
        out["zstd-19"] = (len(zstandard.ZstdCompressor(level=19).compress(data))
                          * 8.0 / len(data))
    except ImportError:
        pass
    try:
        import brotli
        out["brotli-11"] = len(brotli.compress(data, quality=11)) * 8.0 / len(data)
    except ImportError:
        pass
    return out

### Suite B — Correctness gates (lossless is proven, not assumed)

A compressor with a great ratio that cannot reconstruct its input is worthless,
so these are **hard gates** — any FAIL fails the whole benchmark:

1. **Arithmetic-coder self-test** — encode/decode synthetic distributions.
2. **Model-distribution roundtrip** — encode a real batch with the *real*
   causal model probabilities, decode it back, require byte-exact equality.
3. **Tokenizer roundtrip** — 1 MB of enwik8 through BPE encode→decode.
4. **Streaming reference decoder** — a token-by-token decoder (what a receiver
   actually runs) must reproduce the batched distributions.


In [ ]:
# =============================================================================
#  Suite B: correctness gates
# =============================================================================
@torch.no_grad()
def arithmetic_roundtrip(model, tokens, feats, cfg, batch_size):
    """Byte-exact range-coder roundtrip on REAL causal model distributions."""
    loader = eval_loader(tokens, feats, cfg, batch_size)
    batch = move_batch(next(iter(loader)), nsc.DEVICE)
    out = model(batch, stage=2)
    z = out["z_final"][0].float().cpu()
    tgt = batch["target_ids"][0].cpu().numpy()
    probs = torch.softmax(z, -1).numpy()
    cum = [_quantize(probs[t]) for t in range(len(tgt))]
    triples = [(int(cum[t][tgt[t]]), int(cum[t][tgt[t] + 1]), int(cum[t][-1]))
               for t in range(len(tgt))]
    rc = _RangeCoder()
    bits = rc.encode(triples)
    dec = rc.decode(bits, cum, len(tgt))
    return bool(np.array_equal(np.asarray(dec), tgt)), len(bits) / max(1, len(tgt))


def tokenizer_roundtrip(tok, data: bytes) -> bool:
    ids = encode_with(tok, data)
    return decode_with(tok, ids.tolist()) == data


def suite_correctness(model, tok, toks, feats, cfg, batch_size, ref_n):
    print("\n[B] correctness gates ...")
    coder_ok = _selftest_coder(0) and _selftest_coder(7)
    print(f"  arithmetic-coder self-test         : {'PASS' if coder_ok else 'FAIL'}")
    rt_ok, rt_bpt = arithmetic_roundtrip(model, toks["test"], feats["test"], cfg,
                                         batch_size)
    print(f"  model-distribution roundtrip       : {'PASS' if rt_ok else 'FAIL'} "
          f"({rt_bpt:.3f} bits/token achieved)")
    with open(PATHS["data"], "rb") as f:
        tk_ok = tokenizer_roundtrip(tok, f.read(1 << 20))
    print(f"  tokenizer byte-exact roundtrip 1MB : {'PASS' if tk_ok else 'FAIL'}")
    try:
        ref_bpt = reference_decoder_check(model, toks["test"], feats["test"], cfg,
                                          n=min(ref_n, len(toks["test"]) - 2))
        ref_ok = math.isfinite(ref_bpt)
    except Exception as e:
        ref_bpt, ref_ok = float("nan"), False
        print(f"  [warn] reference decoder raised: {e}")
    print(f"  streaming reference decoder        : "
          f"{'PASS' if ref_ok else 'FAIL'} ({ref_bpt:.3f} bits/token)")
    return dict(coder_selftest=coder_ok, roundtrip_lossless=rt_ok,
                roundtrip_bits_per_token=rt_bpt, tokenizer_roundtrip=tk_ok,
                reference_decoder_ok=ref_ok, reference_bits_per_token=ref_bpt,
                lossless=bool(coder_ok and rt_ok and tk_ok))

### Suite C — Calibration ("well calibrated" is part of the question)

Neural nets are usually **overconfident**: they put too much mass on their top
guess. For a compressor that overconfidence is a direct, measurable cost — a
mispredicted token gets an even shorter codeword budget than it deserves, so
surprises cost extra bits. The fix is one scalar: divide the logits by a
temperature `T*` found by golden-section search on validation NLL. Both
encoder and decoder apply the same `T*`, so it is **free at runtime** and worth
≈0.013 bpc here (`T* ≈ 1.11` — confirming mild overconfidence). ECE (expected
calibration error) is reported before/after as the standard diagnostic.


In [ ]:
# =============================================================================
#  Suite C: calibration (RAM-bounded logit sampling)
# =============================================================================
@torch.no_grad()
def suite_calibration(model, loader, max_batches, pos_per_seq, seed=0):
    print("\n[C] calibration ...")
    rng = np.random.RandomState(seed)
    zs, ys = [], []
    for i, batch in enumerate(loader):
        if i >= max_batches:
            break
        batch = move_batch(batch, nsc.DEVICE)
        out = model(batch, stage=2)
        z = out["z_final"].float().cpu()          # [B,T,V]
        y = batch["target_ids"].cpu()
        B, T, V = z.shape
        idx = rng.choice(T, size=min(pos_per_seq, T), replace=False)
        zs.append(z[:, idx].reshape(-1, V))
        ys.append(y[:, idx].reshape(-1))
    z, y = torch.cat(zs), torch.cat(ys)

    def nll(T):
        return torch.nn.functional.cross_entropy(z / T, y).item()

    lo, hi, gr = 0.5, 2.0, (math.sqrt(5) - 1) / 2
    a, b = hi - gr * (hi - lo), lo + gr * (hi - lo)
    fa, fb = nll(a), nll(b)
    for _ in range(40):
        if fa < fb:
            hi, b, fb = b, a, fa
            a = hi - gr * (hi - lo)
            fa = nll(a)
        else:
            lo, a, fa = a, b, fb
            b = lo + gr * (hi - lo)
            fb = nll(b)
        if hi - lo < 1e-3:
            break
    T = 0.5 * (lo + hi)
    res = dict(T_opt=float(T), nll_before=float(nll(1.0)), nll_after=float(nll(T)),
               ece_before=float(nsc._ece(z, y)), ece_after=float(nsc._ece(z / T, y)),
               samples=int(len(y)))
    print(f"  T* = {res['T_opt']:.4f}   NLL {res['nll_before']:.4f} -> "
          f"{res['nll_after']:.4f}   ECE {res['ece_before']:.4f} -> "
          f"{res['ece_after']:.4f}  ({res['samples']:,} sampled positions)")
    return res

### Suite E — Robustness stress tests (where the classics still win)

Three adversarial content types, model vs gzip vs xz on each:

* **repetitive** (`ABCD…`) — LZ codecs collapse this to almost nothing; the
  neural mixer saturates at `match_conf ≈ 0.99`, paying a smoothed tail.
* **random bytes** — *nothing* can compress it; the interesting number is the
  expansion factor. Classical codecs cap near 8.0 bpb; the model currently
  expands badly (no raw-block escape yet — Suite G recommends one).
* **structured text** (a repeated pangram) — same story as repetitive.

This is deliberate honesty: the neural advantage is *domain density*, not
universal dominance, and these rows quantify the failure modes.


In [ ]:
# =============================================================================
#  Suite E: robustness / stress
# =============================================================================
def suite_stress(model, tok, o2, cfg, batch_size, max_batches):
    print("\n[E] robustness stress cases ...")
    cases = {
        "repetitive": b"ABCD" * (1 << 18),                                # 1 MB
        "random": bytes(np.random.RandomState(0)
                        .randint(0, 256, 1 << 17, dtype=np.uint8)),      # 128 KB
        "structured_text": b"the quick brown fox jumps over the lazy dog " * 8192,
    }
    rows = {}
    for name, data in cases.items():
        t = encode_bytes_chunked(tok, data)
        if len(t) < cfg.seg_len + 2:
            rows[name] = {"skipped": f"too short ({len(t)} tokens)"}
            print(f"  {name:<16s} skipped (only {len(t)} tokens)")
            continue
        f = compute_features(t, o2, cfg.order2_match_order, float(cfg.match_len_cap))
        loader = eval_loader(t, f, cfg, batch_size)
        m = eval_metrics(model, loader, len(t), len(data), max_batches=max_batches)
        rows[name] = {"model_bpb": round(m["bits_per_byte"], 4),
                      "gzip_bpb": round(nsc.gzip_bpb(data), 4),
                      "xz_bpb": round(nsc.xz_bpb(data), 4)}
        print(f"  {name:<16s} model {m['bits_per_byte']:7.4f}  "
              f"gzip {rows[name]['gzip_bpb']:7.4f}  xz {rows[name]['xz_bpb']:7.4f}  bpb")
    return rows

### Suite F — Stopping policy (goal · plateau · overtraining)

Answers "should the *training* continue?" with three rules, checked in order:

1. **Overtraining** — do the newest weights do worse on validation than the
   best export by > 0.005 bpb? Then stop and ship `best.pt`.
2. **Goal** — calibrated test bpc ≤ 1.245? Then stop: mission accomplished.
3. **Plateau** — less than ~0.003 bpb gained per ~30k-step session after half
   the curriculum? Then more sessions buy nothing.

Otherwise: CONTINUE, with a decay-model projection of where the curriculum
will land. Note this suite only sees the **two newest checkpoints**; Suite J
below redoes the projection with *every* checkpoint on record, which is the
statistically serious version.


In [ ]:
# =============================================================================
#  Suite F: stopping policy — goal, plateau, overtraining
# =============================================================================
def estimate_final_bpb(step1, bpb1, step2, bpb2, total_steps):
    """Extrapolate the val bpb at curriculum end assuming the improvement rate
    decays linearly to zero at total_steps (conservative cosine-LR heuristic)."""
    if not (math.isfinite(bpb1) and math.isfinite(bpb2)) or step2 <= step1:
        return None
    rate = (bpb1 - bpb2) / (step2 - step1)              # bpb gained per step now
    mid = 0.5 * (step1 + step2)
    if total_steps <= mid or rate <= 0:
        return bpb2
    r0 = rate / (1.0 - mid / total_steps)
    remaining = r0 * (total_steps - step2) ** 2 / (2.0 * total_steps)
    return bpb2 - remaining


def suite_stopping(best_meta, last_meta, val_best, val_last, test_bpb_cal,
                   cfg, goal_bpb):
    print("\n[F] stopping policy ...")
    total_steps = cfg.stage1_steps + cfg.stage2_steps
    completion = best_meta["step"] / max(1, total_steps)

    # overtraining: do the newest weights do WORSE on val than the best export?
    newest, oldest = ((best_meta, last_meta)
                      if best_meta["step"] >= last_meta.get("step", -1)
                      else (last_meta, best_meta))
    val_newest = val_best if newest is best_meta else val_last
    val_prev = val_last if newest is best_meta else val_best
    overfit = (val_last is not None and val_best is not None
               and newest["step"] > oldest.get("step", -1)
               and val_newest > val_prev + OVERFIT_TOL)

    # marginal gain measured between the two checkpoints on record
    s1, b1 = oldest.get("step", -1), oldest.get("val_bpb", float("nan"))
    s2, b2 = newest["step"], newest.get("val_bpb", float("nan"))
    gain = (b1 - b2) if (math.isfinite(b1) and math.isfinite(b2)) else None
    # normalise to a ~1-session stride (30k steps @ ~3k steps/h, 10h cap) so the
    # plateau rule is independent of how far apart the two exports happen to be
    gain_30k = (gain * 30_000.0 / (s2 - s1)) if (gain is not None and s2 > s1) else gain
    proj = estimate_final_bpb(s1, b1, s2, b2, total_steps)

    # calibrated-test projections: test tracks val minus a measured offset
    off = (best_meta["val_bpb"] - test_bpb_cal
           if (test_bpb_cal is not None
               and math.isfinite(best_meta.get("val_bpb", float("nan")))) else None)
    proj_test = (proj - off) if (proj is not None and off is not None) else None
    goal_eta_step = None            # projected step at which the goal is reached
    if (off is not None and math.isfinite(b1) and math.isfinite(b2) and s2 > s1
            and total_steps > 0.5 * (s1 + s2) and b1 > b2):
        rate = (b1 - b2) / (s2 - s1)
        r0 = rate / (1.0 - 0.5 * (s1 + s2) / total_steps)
        tgt_val = goal_bpb + off
        for s in range(int(s2), int(total_steps) + 1, 250):
            if b2 - r0 * ((s - s2) - (s * s - s2 * s2) / (2.0 * total_steps)) <= tgt_val:
                goal_eta_step = s
                break

    plateau_thresh = max(PLATEAU_ABS_BPB, PLATEAU_REL * (b2 if math.isfinite(b2) else 1.3))
    goal_met = test_bpb_cal is not None and test_bpb_cal <= goal_bpb

    if overfit:
        verdict, why = "STOP (overtraining)", (
            f"newest weights (step {newest['step']}) are worse on val than the "
            f"best export by more than {OVERFIT_TOL} bpb — ship best.pt.")
    elif goal_met:
        verdict, why = "STOP (goal reached)", (
            f"calibrated test bpb {test_bpb_cal:.4f} <= goal {goal_bpb}.")
    elif gain_30k is not None and gain_30k < plateau_thresh and completion > 0.5:
        verdict, why = "STOP (plateau)", (
            f"recent gain {gain_30k:+.4f} bpb/session < plateau threshold "
            f"{plateau_thresh:.4f}.")
    else:
        eta_steps = total_steps - newest["step"]
        eta_txt = ("" if goal_eta_step else
                   " The goal is likely decided at curriculum end.")
        verdict, why = "CONTINUE", (
            f"still improving (recent gain "
            f"{gain_30k if gain_30k is None else round(gain_30k, 4)} bpb per "
            f"~30k-step session); {eta_steps:,} curriculum steps remain "
            f"({completion:.0%} complete). Projected final val bpb ≈ {proj:.4f}"
            + (f", calibrated test ≈ {proj_test:.4f}" if proj_test else "")
            + (f"; goal {goal_bpb} projected reachable near step "
               f"{goal_eta_step:,}." if goal_eta_step else "." + eta_txt)
            if proj else "still improving.")

    res = dict(verdict=verdict, reason=why,
               goal_test_bpb=goal_bpb, goal_met=bool(goal_met),
               curriculum_step=newest["step"], curriculum_total=total_steps,
               curriculum_completion=round(completion, 4),
               recent_gain_bpb=gain,
               recent_gain_bpb_per_30k=None if gain_30k is None else round(gain_30k, 4),
               plateau_threshold_bpb=round(plateau_thresh, 4),
               projected_final_val_bpb=None if proj is None else round(proj, 4),
               projected_final_test_bpb_cal=None if proj_test is None else round(proj_test, 4),
               goal_projected_step=goal_eta_step,
               overtraining_detected=bool(overfit),
               measured_val_bpb_best=val_best, measured_val_bpb_last=val_last,
               policy=dict(GOAL_TEST_BPB=goal_bpb, PLATEAU_ABS_BPB=PLATEAU_ABS_BPB,
                           PLATEAU_REL=PLATEAU_REL, OVERFIT_TOL=OVERFIT_TOL,
                           early_stop_patience_in_trainer=cfg.early_stop_patience))
    print(f"  checkpoint step {newest['step']:,}/{total_steps:,} "
          f"({completion:.0%} of curriculum)")
    if gain is not None:
        print(f"  recent val gain {gain:+.4f} bpb over "
              f"{newest['step']-oldest.get('step',0):,} steps "
              f"≈ {gain_30k:+.4f} bpb/session "
              f"(plateau threshold {plateau_thresh:.4f})")
    if proj is not None:
        print(f"  projected final val bpb ≈ {proj:.4f}"
              + (f"   calibrated test ≈ {proj_test:.4f}" if proj_test else "")
              + f" (goal: {goal_bpb})")
    if goal_eta_step is not None and not goal_met:
        print(f"  goal {goal_bpb} projected reachable near step {goal_eta_step:,} "
              f"(~{max(0, goal_eta_step - s2):,} steps away)")
    print(f"  VERDICT: {verdict} — {why}")
    return res

### Suite G — Training state & tuning recommendations

Reconstructs the exact scheduled learning rate at the checkpoint step,
restates val/test losses in nats, bits and bpc, and emits concrete,
condition-triggered recommendations (finish the curriculum, keep the
calibration baked in, add a raw-block escape for incompressible input,
escalate the architecture ladder only after the cosine schedule completes…).
These recommendations are generated from the measurements, not hand-written.


In [ ]:
# =============================================================================
#  Suite G: training state — actual LR / loss + tuning recommendations
# =============================================================================
def lr_at(cfg: Config, step: int) -> float:
    """Reconstruct the trainer's scheduled learning rate at a given step."""
    if step < cfg.stage1_steps:
        return nsc.cosine_lr(step, cfg.stage1_steps, cfg.lr_stage1, cfg.warmup_steps)
    return nsc.cosine_lr(step - cfg.stage1_steps, cfg.stage2_steps,
                         cfg.lr_stage2, cfg.warmup_steps)


def suite_training_state(cfg, best_meta, calib, comp, stress, m_test, m_val,
                         goal_bpb):
    print("\n[G] training state & tuning ...")
    step = best_meta["step"]
    total = cfg.stage1_steps + cfg.stage2_steps
    lr_now = lr_at(cfg, step)
    stage = 1 if step < cfg.stage1_steps else 2
    log2e = math.log2(math.e)
    state = dict(
        step=step, total_steps=total, stage=stage,
        lr_now=lr_now, lr_stage1=cfg.lr_stage1, lr_stage2=cfg.lr_stage2,
        warmup_steps=cfg.warmup_steps,
        val_loss_nats=m_val["nats_per_token"],
        val_loss_bits=m_val["nats_per_token"] * log2e,
        val_bpb=m_val["bits_per_byte"],
        test_loss_nats=m_test["nats_per_token"],
        test_loss_bits=m_test["nats_per_token"] * log2e,
        test_bpb=m_test["bits_per_byte"],
        stored_val_bpb=best_meta.get("val_bpb"),
    )
    print(f"  step {step:,}/{total:,} (stage {stage})   scheduled LR now = {lr_now:.3e}")
    print(f"  val  loss {state['val_loss_nats']:.4f} nats/tok "
          f"({state['val_loss_bits']:.4f} bits/tok, {state['val_bpb']:.4f} bpb)")
    print(f"  test loss {state['test_loss_nats']:.4f} nats/tok "
          f"({state['test_loss_bits']:.4f} bits/tok, {state['test_bpb']:.4f} bpb)")

    recs = []
    if step < total:
        eta_h = (total - step) / 3000.0     # T4 ≈ 3000 optimizer steps/h (base tier)
        recs.append(
            f"Finish the curriculum first — {total - step:,} steps remain "
            f"(~{eta_h:.0f} h on a Kaggle T4 at ≈3,000 steps/h, i.e. "
            f"~{max(1, round(eta_h / 11.5))} more sessions). The cosine schedule "
            f"continues automatically on resume; the LR is already at "
            f"{lr_now:.2e} and anneals to ~0 at step {total:,}, which is itself "
            f"the main protection against late-stage overfitting.")
    gain_cal = comp["test_bpb"] - comp["test_bpb_cal"]
    if calib["T_opt"] > 1.05:
        recs.append(
            f"Logits are overconfident (T* = {calib['T_opt']:.3f}). Keep baking "
            f"calibration_T into the export — it is worth {gain_cal:+.4f} bpb "
            f"for free. If a fresh run is ever started, raising dropout "
            f"{cfg.dropout} → 0.15 or weight_decay {cfg.weight_decay} → 0.02 "
            f"should reduce this overconfidence at the source.")
    rnd = stress.get("random", {})
    if isinstance(rnd.get("model_bpb"), (int, float)) and rnd["model_bpb"] > 8.5:
        recs.append(
            f"Incompressible input EXPANDS badly (random bytes: "
            f"{rnd['model_bpb']:.1f} bpb vs 8.0 raw). Add a raw-block escape to "
            f"the Rust coder: when the running model cost exceeds ~8 bpb over a "
            f"block, emit literal bytes. Caps the worst case at ≈8.03 bpb "
            f"without touching the model.")
    rep = stress.get("repetitive", {})
    if (isinstance(rep.get("model_bpb"), (int, float))
            and rep["model_bpb"] > 20 * max(1e-9, rep.get("xz_bpb", 0))):
        recs.append(
            f"Highly repetitive data: classical LZ crushes it "
            f"(xz {rep.get('xz_bpb')} vs model {rep.get('model_bpb')} bpb). "
            f"The match expert saturates at conf 0.99 with a smoothed tail; a "
            f"cheap LZ pre-pass (or letting match_conf reach ~0.999) would "
            f"close most of that gap.")
    if step >= total:
        recs.append(
            "Curriculum complete. If more budget exists, do NOT keep training "
            "this architecture past the schedule — escalate one ladder row "
            "(d=384, L=5, ~23 M params, rec. 185k steps ≈ 60 h) and reuse the "
            "cached data; more steps on a finished cosine buy almost nothing.")
    recs.append(
        f"Stopping is AUTOMATIC: run_budget() ends the session chain the "
        f"moment a phase confirms calibrated test bpb ≤ {goal_bpb} (the "
        f"statistically projected best-achievable value for this "
        f"architecture), or at curriculum end (step {total:,}), or on a val "
        f"plateau (< {PLATEAU_ABS_BPB} bpb per ~30k-step session; trainer "
        f"patience {cfg.early_stop_patience} evals) — whichever happens "
        f"first. Sessions started after the stop exit immediately.")
    state["recommendations"] = recs
    for i, r in enumerate(recs, 1):
        print(f"  ({i}) {r}")
    return state

### Chart helpers (Part 1)

Shared matplotlib chrome: light surface, hairline solid grid, no top/right
spines, direct labels. The palette is a CVD-validated categorical set (blue =
our model, aqua/yellow = classical context, green = status-good) — the same
system Part 2's dashboard uses, so every figure in the notebook reads as one
family. Charts are saved as PNGs next to the report *and* displayed inline.


In [ ]:
# =============================================================================
#  Charts (matplotlib; light-surface palette, saved as PNG next to the report)
# =============================================================================
PAL = dict(surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781",
           grid="#e1e0d9", axis="#c3c2b7",
           s1="#2a78d6", s2="#1baf7a", s3="#eda100", good="#0ca30c")


def _new_ax(figsize):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    fig.patch.set_facecolor(PAL["surface"])
    ax.set_facecolor(PAL["surface"])
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(PAL["axis"])
        ax.spines[side].set_linewidth(1.0)
    ax.tick_params(colors=PAL["muted"], labelsize=8.5, length=0)
    return fig, ax


def _finish(fig, ax, title, subtitle, path):
    import matplotlib.pyplot as plt
    ax.set_title(title, loc="left", fontsize=11.5, fontweight="semibold",
                 color=PAL["ink"], pad=16)
    if subtitle:
        ax.text(0, 1.015, subtitle, transform=ax.transAxes, fontsize=8.5,
                color=PAL["ink2"], va="bottom")
    fig.tight_layout()
    fig.savefig(path, facecolor=PAL["surface"], bbox_inches="tight")
    plt.close(fig)
    print(f"  [chart] {path}")


def chart_sizes(report, path):
    """Before/after: the 5 MB test corpus vs every compressor's output size."""
    a = report["compression"]
    orig_mb = a["test_bytes"] / 2**20
    rows = [("original (enwik8 test)", orig_mb, PAL["grid"])]
    rows += [(f"hybrid model (ours)", a["test_bpb_cal"] * a["test_bytes"] / 8 / 2**20,
              PAL["s1"])]
    for name, bpb in sorted(a["baselines"].items(), key=lambda kv: kv[1]):
        rows.append((name, bpb * a["test_bytes"] / 8 / 2**20, PAL["axis"]))
    fig, ax = _new_ax((7.4, 0.62 * len(rows) + 1.2))
    y = np.arange(len(rows))[::-1]
    for yi, (name, mb, color) in zip(y, rows):
        ax.barh(yi, mb, height=0.58, color=color, zorder=3)
        ratio = "" if mb >= orig_mb else f"   ({orig_mb / mb:.1f}x smaller)"
        ax.text(mb + orig_mb * 0.012, yi, f"{mb:.2f} MB{ratio}", va="center",
                fontsize=8.5, color=PAL["ink2"])
    ax.set_yticks(y, [r[0] for r in rows], fontsize=9, color=PAL["ink"])
    ax.set_xlim(0, orig_mb * 1.30)
    ax.set_xlabel("size (MB)", fontsize=8.5, color=PAL["muted"])
    ax.xaxis.grid(True, color=PAL["grid"], linewidth=0.8, zorder=0)
    _finish(fig, ax, "File size before and after compression",
            "projected from measured bits/byte on the held-out enwik8 test split",
            path)


def chart_bpb_cases(report, path):
    """Dot plot (log x): bits/byte by content type for model vs gzip vs xz."""
    a = report["compression"]
    cases = [("enwik8 test", a["test_bpb_cal"], a["baselines"].get("gzip-9"),
              a["baselines"].get("xz-9"))]
    for name, row in report["stress"].items():
        if "model_bpb" in row:
            cases.append((name.replace("_", " "), row["model_bpb"],
                          row["gzip_bpb"], row["xz_bpb"]))
    series = [("hybrid (ours)", 1, PAL["s1"]), ("gzip -9", 2, PAL["s2"]),
              ("xz -9", 3, PAL["s3"])]
    fig, ax = _new_ax((7.4, 0.75 * len(cases) + 1.5))
    y = np.arange(len(cases))[::-1]
    for label, col, color in series:
        xs = [max(1e-3, c[col]) for c in cases]
        ax.scatter(xs, y, s=64, color=color, zorder=4, label=label,
                   edgecolors=PAL["surface"], linewidths=2)
    ax.set_xscale("log")
    ax.set_yticks(y, [c[0] for c in cases], fontsize=9, color=PAL["ink"])
    ax.axvline(8.0, color=PAL["axis"], linewidth=1.0, zorder=1)
    ax.text(8.0, y.max() + 0.55, " 8 bpb = raw bytes", fontsize=7.5,
            color=PAL["muted"], va="bottom")
    ax.set_xlabel("bits per byte — lower is better (log scale)",
                  fontsize=8.5, color=PAL["muted"])
    ax.xaxis.grid(True, color=PAL["grid"], linewidth=0.8, zorder=0)
    ax.legend(loc="lower right", frameon=False, fontsize=8.5,
              labelcolor=PAL["ink2"])
    _finish(fig, ax, "Compression by content type",
            "points left of the 8 bpb line compress; right of it expand", path)


def chart_lr(report, cfg, path):
    """The trainer's LR schedule with the checkpoint's position marked."""
    total = cfg.stage1_steps + cfg.stage2_steps
    steps = np.linspace(0, total, 800)
    lrs = np.array([lr_at(cfg, int(s)) for s in steps])
    now = report["checkpoint"]["step"]
    lr_now = lr_at(cfg, now)
    fig, ax = _new_ax((7.4, 3.4))
    ax.plot(steps, lrs * 1e4, color=PAL["s1"], linewidth=2, zorder=3,
            solid_joinstyle="round")
    ax.axvline(cfg.stage1_steps, color=PAL["grid"], linewidth=1.0, zorder=1)
    ax.text(cfg.stage1_steps, ax.get_ylim()[1] * 0.97, " stage 1 → 2",
            fontsize=7.5, color=PAL["muted"], va="top")
    ax.scatter([now], [lr_now * 1e4], s=72, color=PAL["s1"], zorder=5,
               edgecolors=PAL["surface"], linewidths=2)
    ax.annotate(f"now: {lr_now:.2e} @ step {now:,}",
                (now, lr_now * 1e4), xytext=(8, 10),
                textcoords="offset points", fontsize=8.5, color=PAL["ink2"])
    ax.set_xlabel("optimizer step", fontsize=8.5, color=PAL["muted"])
    ax.set_ylabel("learning rate (x 1e-4)", fontsize=8.5, color=PAL["muted"])
    ax.yaxis.grid(True, color=PAL["grid"], linewidth=0.8, zorder=0)
    ax.xaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
    _finish(fig, ax, "Learning-rate schedule (two-stage cosine)",
            f"stage 1: {cfg.lr_stage1:.0e} pretrain - stage 2: "
            f"{cfg.lr_stage2:.0e} residual fine-tune - anneals to ~0 at "
            f"step {total // 1000}k", path)


def chart_trajectory(report, cfg, goal_bpb, path):
    """Observed val bpb + decay-model projection to curriculum end + goal."""
    total = cfg.stage1_steps + cfg.stage2_steps
    pts = []
    for key in ("latest_checkpoint", "checkpoint"):
        m = report.get(key) or {}
        if m.get("step", -1) > 0 and math.isfinite(m.get("val_bpb", float("nan"))):
            pts.append((m["step"], m["val_bpb"]))
    pts = sorted(set(pts))
    fig, ax = _new_ax((7.4, 3.6))
    if len(pts) >= 2:
        (s1, b1), (s2, b2) = pts[0], pts[-1]
        rate = (b1 - b2) / max(1, s2 - s1)
        mid = 0.5 * (s1 + s2)
        r0 = rate / max(1e-9, 1.0 - mid / total)
        proj_s = np.linspace(s2, total, 200)
        proj_b = b2 - r0 * ((proj_s - s2) - (proj_s**2 - s2**2) / (2 * total))
        ax.plot(proj_s, proj_b, color=PAL["s1"], linewidth=2, linestyle=(0, (4, 3)),
                zorder=3)
        ax.annotate(f"projected ≈ {proj_b[-1]:.4f}", (total, proj_b[-1]),
                    xytext=(-8, -14), textcoords="offset points", ha="right",
                    fontsize=8.5, color=PAL["ink2"])
    xs, ys = zip(*pts) if pts else ((), ())
    ax.plot(xs, ys, color=PAL["s1"], linewidth=2, zorder=4)
    ax.scatter(xs, ys, s=64, color=PAL["s1"], zorder=5,
               edgecolors=PAL["surface"], linewidths=2)
    for s, b in pts:
        ax.annotate(f"{b:.4f}", (s, b), xytext=(0, 9), ha="center",
                    textcoords="offset points", fontsize=8, color=PAL["ink2"])
    ax.axhline(goal_bpb, color=PAL["good"], linewidth=1.4, zorder=2)
    ax.text(0.01, goal_bpb, f"goal: {goal_bpb} bpb — stop here",
            transform=ax.get_yaxis_transform(), fontsize=8, color="#006300",
            va="bottom")
    ax.set_xlim(min(xs) - 4000 if xs else 0, total + 6000)
    ax.set_xlabel("optimizer step", fontsize=8.5, color=PAL["muted"])
    ax.set_ylabel("validation bits/byte", fontsize=8.5, color=PAL["muted"])
    ax.yaxis.grid(True, color=PAL["grid"], linewidth=0.8, zorder=0)
    ax.xaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
    _finish(fig, ax, "Validation bits/byte — observed, projected, and the goal",
            "solid: checkpoints on record - dashed: linear-decay projection to "
            "curriculum end", path)


def make_charts(report, cfg, goal_bpb, outdir):
    try:
        import matplotlib
        matplotlib.use("Agg")
    except ImportError:
        print("  [charts] matplotlib not installed — skipping charts")
        return {}
    cdir = os.path.join(outdir, "charts")
    os.makedirs(cdir, exist_ok=True)
    print("\n[charts] rendering ...")
    jobs = {"sizes": lambda p: chart_sizes(report, p),
            "bpb_cases": lambda p: chart_bpb_cases(report, p),
            "lr_schedule": lambda p: chart_lr(report, cfg, p),
            "val_trajectory": lambda p: chart_trajectory(report, cfg, goal_bpb, p)}
    out = {}
    for name, fn in jobs.items():
        path = os.path.join(cdir, f"{name}.png")
        try:
            fn(path)
            out[name] = os.path.join("charts", f"{name}.png")
        except Exception as e:
            print(f"  [charts] {name} failed (non-fatal): {e}")
    return out

### Report writer

Serializes everything into `benchmark_report.json` (machine-readable; Part 2
and CI consume this) and `benchmark_report.md` (human-readable summary with
the charts embedded). Written next to `results/`, or to `./benchmark_out` when
inputs are read-only (Kaggle datasets).


In [ ]:
# =============================================================================
#  Report
# =============================================================================
def write_report(report, outdir):
    jpath = os.path.join(outdir, "benchmark_report.json")
    with open(jpath, "w") as f:
        json.dump(report, f, indent=2, default=float)

    a, b_, c = report["compression"], report["correctness"], report["calibration"]
    d, f_, env = report["performance"], report["stopping"], report["environment"]
    g, charts = report.get("training_state", {}), report.get("charts", {})
    test_mb = a.get("test_bytes", 0) / 2**20
    lines = [
        "# Enterprise benchmark report", "",
        f"*Generated {report['generated']} — checkpoint "
        f"`{os.path.basename(report['checkpoint']['path'])}` step "
        f"{report['checkpoint']['step']:,} — device {env['device']}*", "",
        "## A. Compression quality (test split)", "",
        "| metric | value |", "|---|---|",
        f"| bits/byte (raw) | {a['test_bpb']:.4f} |",
        f"| bits/byte (calibrated T*={c['T_opt']:.3f}) | {a['test_bpb_cal']:.4f} |",
        f"| perplexity | {a['perplexity']:.2f} |",
        f"| top-1 / top-5 accuracy | {a['top1_acc']:.2%} / {a['top5_acc']:.2%} |",
        f"| projected size of 100 MB enwik8 | {a['projected_mb_per_100mb']:.2f} MB |",
        f"| eval coverage | {a['coverage_frac']:.1%} of test split |", "",
        "### Before / after (5 MB held-out corpus)", "",
        "| compressor | size after | ratio | bits/byte |", "|---|---|---|---|",
        f"| — original | {test_mb:.2f} MB | 1.00x | 8.0000 |",
        f"| **hybrid model (ours)** | "
        f"**{a['test_bpb_cal'] * a.get('test_bytes', 0) / 8 / 2**20:.2f} MB** | "
        f"**{8.0 / a['test_bpb_cal']:.2f}x** | **{a['test_bpb_cal']:.4f}** |"]
    for name, bpb in sorted(a["baselines"].items(), key=lambda kv: kv[1]):
        lines.append(f"| {name} | {bpb * a.get('test_bytes', 0) / 8 / 2**20:.2f} MB "
                     f"| {8.0 / bpb:.2f}x | {bpb:.4f} |")
    if charts.get("sizes"):
        lines += ["", f"![before/after sizes]({charts['sizes']})"]
    lines += [
        "", "## B. Correctness gates", "",
        f"- arithmetic-coder self-test: **{'PASS' if b_['coder_selftest'] else 'FAIL'}**",
        f"- model-distribution lossless roundtrip: "
        f"**{'PASS' if b_['roundtrip_lossless'] else 'FAIL'}** "
        f"({b_['roundtrip_bits_per_token']:.3f} bits/token)",
        f"- tokenizer byte-exact roundtrip: "
        f"**{'PASS' if b_['tokenizer_roundtrip'] else 'FAIL'}**",
        f"- streaming reference decoder: "
        f"**{'PASS' if b_['reference_decoder_ok'] else 'FAIL'}**", "",
        "## C. Calibration", "",
        f"- optimal temperature T* = {c['T_opt']:.4f}",
        f"- NLL {c['nll_before']:.4f} → {c['nll_after']:.4f}",
        f"- ECE {c['ece_before']:.4f} → {c['ece_after']:.4f}", "",
        "## D. Performance", "",
        f"- parameters: {d['params_m']:.2f} M  (checkpoint "
        f"{report['checkpoint']['size_mb']:.0f} MB on disk)",
        f"- throughput: {d['throughput_tok_s']:.0f} tok/s ≈ "
        f"{d['throughput_mb_s']:.3f} MB/s  ({env['device']}, batched upper bound)",
        f"- latency per batch p50/p95: {d['latency_ms_p50']:.0f} / "
        f"{d['latency_ms_p95']:.0f} ms",
        f"- peak RSS: {d['peak_rss_gb']:.2f} GB", "",
        "## E. Robustness", "",
        "| case | model bpb | gzip bpb | xz bpb |", "|---|---|---|---|"]
    for name, row in report["stress"].items():
        if "skipped" in row:
            lines.append(f"| {name} | skipped | | |")
        else:
            lines.append(f"| {name} | {row['model_bpb']} | {row['gzip_bpb']} | "
                         f"{row['xz_bpb']} |")
    if charts.get("bpb_cases"):
        lines += ["", f"![bpb by content type]({charts['bpb_cases']})"]
    lines += [
        "", "## F. Stopping policy", "",
        f"**{f_['verdict']}** — {f_['reason']}", "",
        f"- curriculum: step {f_['curriculum_step']:,} / "
        f"{f_['curriculum_total']:,} ({f_['curriculum_completion']:.0%})",
        f"- goal (calibrated test bpb): {f_['goal_test_bpb']} — "
        f"{'met' if f_['goal_met'] else 'not met'}",
        f"- projected final val bpb: {f_['projected_final_val_bpb']}",
        f"- projected final calibrated test bpb: "
        f"{f_.get('projected_final_test_bpb_cal')}",
        f"- goal projected reachable near step: "
        f"{f_.get('goal_projected_step') or '— (curriculum end decides)'}",
        f"- overtraining detected: {f_['overtraining_detected']}"]
    if charts.get("val_trajectory"):
        lines += ["", f"![validation trajectory]({charts['val_trajectory']})"]
    if g:
        lines += [
            "", "## G. Training state & tuning", "",
            "| quantity | value |", "|---|---|",
            f"| optimizer step | {g['step']:,} / {g['total_steps']:,} "
            f"(stage {g['stage']}) |",
            f"| scheduled LR now | {g['lr_now']:.3e} |",
            f"| LR plan | stage 1: {g['lr_stage1']:.0e} → stage 2: "
            f"{g['lr_stage2']:.0e}, warmup {g['warmup_steps']:,}, cosine → 0 |",
            f"| val loss | {g['val_loss_nats']:.4f} nats/tok = "
            f"{g['val_loss_bits']:.4f} bits/tok ({g['val_bpb']:.4f} bpb) |",
            f"| test loss | {g['test_loss_nats']:.4f} nats/tok = "
            f"{g['test_loss_bits']:.4f} bits/tok ({g['test_bpb']:.4f} bpb) |",
            f"| stored best val bpb (full split) | {g['stored_val_bpb']:.4f} |"]
        if charts.get("lr_schedule"):
            lines += ["", f"![LR schedule]({charts['lr_schedule']})"]
        lines += ["", "### Recommendations", ""]
        lines += [f"{i}. {r}" for i, r in enumerate(g.get("recommendations", []), 1)]
    lines += ["", f"**Overall: {'PASS' if report['gates_passed'] else 'FAIL'}**", ""]
    mpath = os.path.join(outdir, "benchmark_report.md")
    with open(mpath, "w") as f:
        f.write("\n".join(lines))
    return jpath, mpath

### Main — the orchestration (definitions only; the next cell runs it)

Wires the suites together in dependency order: load checkpoint → prepare
splits/features (cached) → A quality → C calibration → recompute calibrated
quality → B correctness → D performance → E stress → F stopping (evaluating
the `.last` checkpoint too, for the overtraining check) → G training state →
gates → charts → report. Returns a CI-style exit code (0 = all hard gates
passed).


In [ ]:
# =============================================================================
#  Main
# =============================================================================
PATHS = {}


def main(argv=None):
    """Run the benchmark. `argv=None` parses sys.argv (CLI); pass a list (e.g.
    [] or ["--full"]) when calling from a notebook cell — unknown kernel args
    are ignored."""
    global PATHS
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--train", action="store_true",
                    help="run a training session first (overrides SKIP_TRAINING=True)")
    ap.add_argument("--budget", type=float, default=TRAIN_BUDGET_H,
                    help="total cross-session budget in hours (training mode)")
    ap.add_argument("--session-cap", type=float, default=SESSION_CAP_H)
    ap.add_argument("--export-dir", default=None,
                    help="dir with best.pt + tokenizer.json (default: results/export)")
    ap.add_argument("--data", default=None, help="path to enwik8")
    ap.add_argument("--full", action="store_true",
                    help="evaluate the FULL val/test splits (slow on CPU)")
    ap.add_argument("--max-batches", type=int, default=TEST_MAX_BATCHES)
    ap.add_argument("--goal-bpb", type=float, default=GOAL_TEST_BPB)
    ap.add_argument("--skip-last-ckpt", action="store_true",
                    help="skip evaluating the .last checkpoint (halves RAM/time)")
    args, _unknown = ap.parse_known_args(argv)

    skip_training = SKIP_TRAINING and not args.train
    if not skip_training:
        print("[mode] TRAINING ENABLED — running a session via run_budget() first")
        nsc.run_budget(total_budget_h=args.budget, session_cap_h=args.session_cap)
        runs = os.path.join(nsc.WORKDIR, "runs")
        cands = sorted((os.path.join(r, "best.pt")
                        for r in (os.path.join(runs, d) for d in os.listdir(runs))
                        if os.path.exists(os.path.join(r, "best.pt"))),
                       key=os.path.getmtime)
        if cands:
            args.export_dir = os.path.dirname(cands[-1])
    else:
        print("[mode] SKIP_TRAINING=True — evaluating previously exported results only")

    PATHS = locate_paths(args.export_dir, args.data)
    print(f"[paths] export={PATHS['export_dir']}")
    print(f"[paths] ckpt  ={PATHS['best_ckpt']}  last={PATHS['last_ckpt']}")
    print(f"[paths] data  ={PATHS['data']}")

    test_cap = None if args.full else args.max_batches
    val_cap = None if args.full else VAL_MAX_BATCHES

    # ---- load model + tokenizer + data ----
    state, cfg_dict, best_meta = load_checkpoint_meta(PATHS["best_ckpt"])
    model, cfg = build_model(state, cfg_dict)
    del state
    print(f"[model] tier={cfg.tier} d={cfg.d_model} L={cfg.n_layers} "
          f"vocab={cfg.vocab_size} params={count_params(model)/1e6:.2f}M "
          f"step={best_meta['step']:,} stored val_bpb={best_meta['val_bpb']:.4f}")

    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(PATHS["tokenizer"])
    assert tok.get_vocab_size() == cfg.vocab_size, "tokenizer/config vocab mismatch"

    cache_dir = writable_dir(os.path.join(os.path.dirname(PATHS["export_dir"]),
                                          "cache"))
    print("\n[data] preparing splits + classical features (cached) ...")
    byts, toks, feats, o2 = prepare_eval_data(tok, cfg, PATHS["data"], cache_dir)

    va_loader = eval_loader(toks["val"], feats["val"], cfg, EVAL_BATCH_SIZE)
    te_loader = eval_loader(toks["test"], feats["test"], cfg, EVAL_BATCH_SIZE)

    # ---- Suite A: compression quality ----
    print(f"\n[A] compression quality on test split "
          f"({'FULL' if test_cap is None else f'{test_cap} batches'}) ...")
    m_test = eval_metrics(model, te_loader, len(toks["test"]), len(byts["test"]),
                          max_batches=test_cap)
    print(f"  raw test bpb {m_test['bits_per_byte']:.4f}   ppl "
          f"{m_test['perplexity']:.2f}   top1 {m_test['top1_acc']:.2%}   "
          f"top5 {m_test['top5_acc']:.2%}   ({m_test['tokens_evaluated']:,} tokens, "
          f"{m_test['coverage_frac']:.1%} of split)")
    print("  classical baselines on the same bytes ...")
    baselines = baseline_bpb(byts["test"])
    for k, v in baselines.items():
        print(f"    {k:<10s} {v:.4f} bpb")

    # ---- Suite C: calibration, then calibrated test bpb ----
    calib = suite_calibration(model, va_loader, CALIB_BATCHES, CALIB_POS_PER_SEQ)
    T = calib["T_opt"]
    m_test_cal = eval_metrics(model, te_loader, len(toks["test"]), len(byts["test"]),
                              max_batches=test_cap, temperature=T)
    print(f"  calibrated test bpb {m_test_cal['bits_per_byte']:.4f} (T*={T:.3f})")

    comp = dict(test_bpb=m_test["bits_per_byte"],
                test_bpb_cal=m_test_cal["bits_per_byte"],
                perplexity=m_test["perplexity"], top1_acc=m_test["top1_acc"],
                top5_acc=m_test["top5_acc"], coverage_frac=m_test["coverage_frac"],
                tokens_evaluated=m_test["tokens_evaluated"],
                test_bytes=len(byts["test"]),
                projected_mb_per_100mb=m_test_cal["bits_per_byte"] * 100 / 8,
                baselines=baselines)

    # ---- Suite B: correctness ----
    corr = suite_correctness(model, tok, toks, feats, cfg, EVAL_BATCH_SIZE,
                             REF_DECODER_N)

    # ---- Suite D: performance ----
    perf = dict(params_m=count_params(model) / 1e6,
                throughput_tok_s=m_test["throughput_tok_s"],
                throughput_mb_s=(m_test["throughput_tok_s"]
                                 * len(byts["test"]) / len(toks["test"]) / 2**20),
                latency_ms_p50=m_test["latency_ms_p50"],
                latency_ms_p95=m_test["latency_ms_p95"],
                peak_rss_gb=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20)
    print(f"\n[D] performance: {perf['throughput_tok_s']:.0f} tok/s "
          f"({perf['throughput_mb_s']:.3f} MB/s), batch latency p50 "
          f"{perf['latency_ms_p50']:.0f} ms, peak RSS {perf['peak_rss_gb']:.2f} GB")

    # ---- Suite E: stress ----
    stress = suite_stress(model, tok, o2, cfg, EVAL_BATCH_SIZE, STRESS_MAX_BATCHES)

    # ---- Suite F: stopping policy (compare best vs latest checkpoint) ----
    m_val = eval_metrics(model, va_loader, len(toks["val"]), len(byts["val"]),
                         max_batches=val_cap)
    val_best = m_val["bits_per_byte"]
    last_meta, val_last = {}, None
    if PATHS["last_ckpt"] and not args.skip_last_ckpt:
        del model
        state, cfg2, last_meta = load_checkpoint_meta(PATHS["last_ckpt"])
        model2, _ = build_model(state, cfg2)
        del state
        val_last = eval_metrics(model2, va_loader, len(toks["val"]),
                                len(byts["val"]), max_batches=val_cap)["bits_per_byte"]
        del model2
        print(f"\n  measured val bpb — best.pt(step {best_meta['step']:,}): "
              f"{val_best:.4f}   {os.path.basename(last_meta['path'])}"
              f"(step {last_meta['step']:,}): {val_last:.4f}")
    stop = suite_stopping(best_meta, last_meta or {"step": -1}, val_best, val_last,
                          comp["test_bpb_cal"], cfg, args.goal_bpb)

    # ---- Suite G: training state (actual LR / loss) + tuning advice ----
    tstate = suite_training_state(cfg, best_meta, calib, comp, stress,
                                  m_test, m_val, args.goal_bpb)

    # ---- gates + report ----
    gates = dict(lossless=corr["lossless"],
                 reference_decoder=corr["reference_decoder_ok"],
                 beats_gzip=comp["test_bpb_cal"] < baselines["gzip-9"],
                 calibration_finite=math.isfinite(calib["T_opt"]))
    passed = all(gates.values())
    report = dict(
        generated=time.strftime("%Y-%m-%d %H:%M:%S"),
        environment=dict(python=platform.python_version(),
                         torch=torch.__version__,
                         device=str(nsc.DEVICE), host=platform.node()),
        checkpoint=best_meta, latest_checkpoint=last_meta,
        config={k: cfg_dict[k] for k in ("tier", "vocab_size", "d_model", "n_layers",
                                         "n_heads", "gru_hidden", "seg_len",
                                         "stage1_steps", "stage2_steps")},
        compression=comp, correctness=corr, calibration=calib, performance=perf,
        stress=stress, stopping=stop, training_state=tstate,
        gates=gates, gates_passed=passed)
    outdir = writable_dir(os.path.dirname(PATHS["export_dir"]))
    report["charts"] = make_charts(report, cfg, args.goal_bpb, outdir)
    globals().update(_LAST_CHARTS=report["charts"], _LAST_OUTDIR=outdir)
    jpath, mpath = write_report(report, outdir)

    print("\n" + "=" * 78)
    print(f"  OVERALL: {'PASS' if passed else 'FAIL'}   "
          f"test bpb {comp['test_bpb_cal']:.4f} (calibrated)   "
          f"gzip {baselines['gzip-9']:.3f}   xz {baselines['xz-9']:.3f}")
    print(f"  STOPPING: {stop['verdict']}")
    print(f"  report: {jpath}\n          {mpath}")
    print("=" * 78)
    return 0 if passed else 1

### ▶ RUN CELL — Part 1's single heavy cell

Edit `FLAGS` if needed, then run. Useful variants:

| FLAGS | Effect |
|---|---|
| `[]` | capped eval — CPU-friendly, slice-noise caveat applies |
| `["--full"]` | full val/test splits — confirmatory numbers (use GPU) |
| `["--goal-bpb", "1.28"]` | try a different stopping goal |
| `["--export-dir", "/kaggle/input/<ds>/export"]` | explicit artifact dir |
| `["--skip-last-ckpt"]` | halve RAM/time (skips the overtraining check) |

**Already have a saved report?** Skip this cell entirely — Part 2 below
auto-loads the newest `benchmark_report.json` it can find.


In [ ]:
# =============================================================================
#  RUN CELL -- execute the benchmark (edit FLAGS, then run)
#
#  SKIP_TRAINING=True (Section 1 CONFIG) evaluates the exported previous-session
#  results only. FLAGS mirror the CLI of benchmark_enterprise.py:
#      []                                   capped eval (CPU-friendly defaults)
#      ["--full"]                           full val/test splits (use on GPU)
#      ["--goal-bpb", "1.20"]               change the stopping goal
#      ["--export-dir", "/kaggle/input/<ds>/export"]   explicit artifact dir
#      ["--train", "--budget", "80"]        run a training session FIRST, then
#                                           benchmark its freshest checkpoint
#
#  Outputs: benchmark_report.json / benchmark_report.md / charts/*.png
#  (written next to results/, or to ./benchmark_out on read-only inputs).
# =============================================================================
FLAGS: list = []

rc = main(FLAGS)
print("\nexit code:", rc, "(0 = all correctness gates passed)")

# Show the charts inline
from IPython.display import Image, display
for _name, _rel in (globals().get("_LAST_CHARTS") or {}).items():
    _p = os.path.join(_LAST_OUTDIR, _rel)
    if os.path.exists(_p):
        display(Image(filename=_p))


---
## Part 2 — The verdict: head-to-head against the traditional methods

Suite A showed the model needs *fewer bits* than gzip/xz. That alone does not
make it "superior" — a method that saves 40 % of the bits but runs 10,000×
slower and needs a 200 MB decoder is only superior in the situations where
those costs don't bind. So Part 2 measures **all four axes on equal terms**
and then—only then—answers the question.

**Protocol (fairness rules).**

* Same bytes for everyone: the identical 5 MB held-out test split Suite A
  scored the model on. Same machine, wall-clock timed.
* Every classical codec runs at its **best practical setting** (`gzip -9`,
  `bzip2 -9`, `xz -9`, `zstd -19`, `brotli -11`, `7z PPMd o16` — PPMd being
  the strongest *traditional statistical* method, the model's closest classical
  relative), **single-threaded**, in a **fresh child process** so peak RSS is
  clean, and must pass a **byte-exact roundtrip** — the same standard Suite B
  holds the model to.
* Sub-second timings are re-run 3× (median kept).
* The model's compress speed is its **teacher-forced batched bound** from
  Suite D — the most favorable defensible number (only the *sender* can batch
  like that). Its decompress speed is **measured directly** below with a
  token-by-token windowed decoder, the workload a receiver cannot avoid; a
  production incremental decoder (the exported ONNX step graphs) lands between
  the two bounds. Both labels appear on the charts.


### Suite H — the measurements

Prints one row per codec: `bpc` (density), compress / decompress MB/s, CPU
core-seconds per 100 MB, peak RSS. Also measures the **decoder footprint** —
everything a receiver must hold to decompress: ONNX step graphs, the order-0/1
tables, the order-2 table and the tokenizer. Classical codecs carry their
decoder in `libc`-sized binaries (≈0 by comparison); this asymmetry is priced
into the break-even analysis, and is the fair way to compare "size".
Results are saved to `head_to_head.json`.


In [ ]:
# =============================================================================
#  SECTION 8 -- SUITE H: HEAD-TO-HEAD vs CLASSICAL CODECS (measured, same bytes)
# =============================================================================
#  Suite A only records the classical compressors' RATIO. To answer "is the
#  neural approach actually superior, and at what price?" we also need their
#  SPEED, MEMORY and CPU COST, measured on the SAME 5 MB held-out test split,
#  on THIS machine. Protocol:
#    * every classical codec runs in a fresh child process (clean peak-RSS
#      reading, no allocator pollution between codecs), single-threaded
#      (-T1 / -mmt=1) so "seconds" mean "CPU-core-seconds";
#    * compress AND decompress are timed; the roundtrip must be byte-exact
#      (same hard gate the model faces in Suite B);
#    * sub-second codecs are timed 3x and the median is kept;
#    * the model's numbers come from the newest benchmark_report.json (Suite
#      A/D) plus an optional direct measurement of true streaming decode below.
#  Optional codecs (zstd CLI, 7z PPMd, python-brotli) are skipped gracefully
#  when not installed, locally and on Kaggle alike.
# =============================================================================
import subprocess, shutil, tempfile, statistics, importlib.util

MEASURE_STREAM_DECODE = True   # time the token-by-token reference decoder
STREAM_DECODE_TOKENS  = 1024   # tokens to time (incremental decoder — a full window)
STREAM_DECODE_BUDGET_S = 45.0  # hard wall-clock cap for that measurement

# ---- find the newest benchmark_report.json (this session's, or a saved one) --
def load_latest_report():
    cands = []
    if globals().get("_LAST_OUTDIR"):
        cands.append(os.path.join(_LAST_OUTDIR, "benchmark_report.json"))
    for pat in (os.path.join(BASE_DIR, "results", "runs", "benchmark_report.json"),
                os.path.join(BASE_DIR, "results", "benchmark_report.json"),
                os.path.join(BASE_DIR, "benchmark_out", "benchmark_report.json"),
                os.path.join(os.getcwd(), "benchmark_out", "benchmark_report.json"),
                "/kaggle/working/**/benchmark_report.json",
                "/kaggle/input/*/benchmark_report.json",
                "/kaggle/input/*/*/benchmark_report.json"):
        cands += glob.glob(pat, recursive=True)
    cands = [p for p in dict.fromkeys(cands) if os.path.exists(p)]
    if not cands:
        raise FileNotFoundError(
            "no benchmark_report.json found — run the RUN CELL (main) once, "
            "or attach a dataset that contains a previous report.")
    path = max(cands, key=os.path.getmtime)
    with open(path) as f:
        rep = json.load(f)
    print(f"[H] using report {path}  (generated {rep.get('generated')}, "
          f"checkpoint step {rep.get('checkpoint', {}).get('step')})")
    return rep, path


def hh_outdir(report_path):
    return writable_dir(os.path.dirname(os.path.abspath(report_path)))


# ---- rebuild the exact test-split bytes the report was scored on ------------
def hh_test_bytes():
    global PATHS
    if not PATHS:
        PATHS = locate_paths()
    cfg0 = Config()                       # split fractions are Config defaults
    with open(PATHS["data"], "rb") as f:
        data = f.read()
    n = len(data)
    n_tr, n_va = int(n * cfg0.train_frac), int(n * cfg0.val_frac)
    return data[n_tr + n_va:]


# ---- one codec, one fresh child process -------------------------------------
# The child prints a single JSON line: it times compress + decompress on the
# input file, verifies the roundtrip byte-exactly, and self-reports peak RSS
# (its own for in-process codecs, its subprocess's for CLI codecs).
_HH_CHILD = r'''
import sys, os, json, time, resource, subprocess, tempfile
inp, codec = sys.argv[1], sys.argv[2]
data = open(inp, "rb").read()
res = dict(codec=codec)
def emit(**kw):
    res.update(kw); print("@@HH@@" + json.dumps(res)); sys.exit(0)
def peak_mb(children=False):
    # ru_maxrss survives execve on Linux (inherits the launcher's watermark),
    # so in-process peaks come from VmHWM; CLI peaks from RUSAGE_CHILDREN.
    if not children:
        try:
            with open("/proc/self/status") as f:
                for line in f:
                    if line.startswith("VmHWM"):
                        return int(line.split()[1]) / 1024.0
        except OSError:
            pass
    who = resource.RUSAGE_CHILDREN if children else resource.RUSAGE_SELF
    return resource.getrusage(who).ru_maxrss / 1024.0
try:
    if codec.startswith("cli:"):
        _, tool, comp_args, dec_args = codec.split("|")
        with tempfile.TemporaryDirectory() as td:
            cf, df = os.path.join(td, "c.bin"), os.path.join(td, "d.bin")
            t0 = time.perf_counter()
            subprocess.run([tool] + comp_args.split() + [cf, inp],
                           check=True, capture_output=True)
            tc = time.perf_counter() - t0
            t0 = time.perf_counter()
            subprocess.run([tool] + dec_args.split() + [df, cf],
                           check=True, capture_output=True)
            td_ = time.perf_counter() - t0
            ok = open(df, "rb").read() == data
            out_bytes = os.path.getsize(cf)
        rss = peak_mb(children=True)
    elif codec.startswith("7z:"):
        with tempfile.TemporaryDirectory() as td:
            cf = os.path.join(td, "c.7z")
            t0 = time.perf_counter()
            subprocess.run(["7z", "a", "-bd", "-y", "-mmt=1",
                            "-m0=" + codec.split(":", 1)[1], cf, inp],
                           check=True, capture_output=True)
            tc = time.perf_counter() - t0
            t0 = time.perf_counter()
            p = subprocess.run(["7z", "e", "-bd", "-y", "-so", cf],
                               check=True, capture_output=True)
            td_ = time.perf_counter() - t0
            ok = p.stdout == data
            out_bytes = os.path.getsize(cf)
        rss = peak_mb(children=True)
    else:
        if codec == "gzip-9":
            import gzip; c = lambda d: gzip.compress(d, 9); x = gzip.decompress
        elif codec == "gzip-6":
            import gzip; c = lambda d: gzip.compress(d, 6); x = gzip.decompress
        elif codec == "bzip2-9":
            import bz2; c = lambda d: bz2.compress(d, 9); x = bz2.decompress
        elif codec == "xz-9":
            import lzma; c = lambda d: lzma.compress(d, preset=9); x = lzma.decompress
        elif codec == "brotli-11":
            import brotli; c = lambda d: brotli.compress(d, quality=11); x = brotli.decompress
        elif codec == "zstd-19":
            import zstandard as z
            c = lambda d: z.ZstdCompressor(level=19).compress(d)
            x = lambda b: z.ZstdDecompressor().decompress(b)
        else:
            emit(ok=False, error="unknown codec")
        t0 = time.perf_counter(); blob = c(data); tc = time.perf_counter() - t0
        t0 = time.perf_counter(); back = x(blob); td_ = time.perf_counter() - t0
        ok, out_bytes = back == data, len(blob)
        rss = peak_mb()
    emit(ok=bool(ok), comp_s=tc, decomp_s=td_, out_bytes=out_bytes, maxrss_mb=rss)
except Exception as e:
    emit(ok=False, error=str(e)[:200])
'''


def _hh_run_child(codec, inp_path):
    p = subprocess.run([sys.executable, "-c", _HH_CHILD, inp_path, codec],
                       capture_output=True, text=True, timeout=1800)
    for line in p.stdout.splitlines()[::-1]:
        if line.startswith("@@HH@@"):
            return json.loads(line[6:])
    return dict(ok=False, error=(p.stderr or "no output")[:200])


def _hh_available_codecs():
    """(label, child-spec) pairs, worst-ratio-first; missing tools are skipped."""
    have_py = lambda m: importlib.util.find_spec(m) is not None
    out = [("gzip-6 (everyday default)", "gzip-6"),
           ("gzip-9", "gzip-9"),
           ("bzip2-9", "bzip2-9"),
           ("xz-9 (LZMA)", "xz-9")]
    if have_py("zstandard"):
        out.append(("zstd-19", "zstd-19"))
    elif shutil.which("zstd"):
        out.append(("zstd-19", "cli:|zstd|-19 -T1 -q -f -o|-d -q -f -o"))
    if have_py("brotli"):
        out.append(("brotli-11", "brotli-11"))
    if shutil.which("7z"):
        out.append(("7z-PPMd o16 (best classical statistical)",
                    "7z:PPMd:o16:mem192m"))
    return out


# ---- true streaming decode speed (production incremental decoder) -----------
@torch.no_grad()
def measure_streaming_decode(n_tokens=STREAM_DECODE_TOKENS,
                             budget_s=STREAM_DECODE_BUDGET_S):
    """Token-by-token decode exactly as a receiver must do it, using the
    PRODUCTION incremental decoder (nsc.StreamingStepper): GRU hidden state +
    transformer KV cache carried across tokens and reset at each seg_len window
    boundary — O(1) GRU + O(context) attention per token. This is the honest
    counterpart of the batched (teacher-forced) throughput in Suite D, which
    only the SENDER can use. (The old window re-forward — one full seg_len
    forward per emitted token — is ~100x slower and survives only as the
    correctness oracle reference_decoder_check.)"""
    global PATHS
    if not PATHS:
        PATHS = locate_paths()
    state, cfg_dict, meta = load_checkpoint_meta(PATHS["best_ckpt"])
    model, cfg = build_model(state, cfg_dict)
    del state
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(PATHS["tokenizer"])
    cache_dir = writable_dir(os.path.join(os.path.dirname(PATHS["export_dir"]),
                                          "cache"))
    _, toks, feats, _ = prepare_eval_data(tok, cfg, PATHS["data"], cache_dir)
    t, fe = toks["test"], feats["test"]
    seg, i0 = cfg.seg_len, 2 * cfg.seg_len         # start in steady-state region
    n_tokens = max(1, min(n_tokens, len(t) - 1 - i0))
    stepper = nsc.StreamingStepper(model, cfg, T=1.0)
    done, t0 = 0, time.perf_counter()
    for k in range(n_tokens):
        i = i0 + k
        if i % seg == 0:
            stepper.reset_window()                 # decode resets every window
        stepper.step(int(t[i]), int(fe["order2_top"][i]),
                     float(fe["order2_conf"][i]), int(fe["match_top"][i]),
                     float(fe["match_conf"][i]))
        done += 1
        if time.perf_counter() - t0 > budget_s:
            break
    dt = time.perf_counter() - t0
    tok_s = done / max(1e-9, dt)
    bpt = len(hh_test_bytes()) / max(1, len(t))    # bytes per token on test
    del model
    return dict(tokens=done, seconds=round(dt, 2), tok_s=tok_s,
                bytes_per_token=bpt, mb_s=tok_s * bpt / 2**20,
                window=seg, note="production incremental decoder "
                "(nsc.StreamingStepper: GRU state + transformer KV cache, reset "
                "per seg_len window); decode is inherently sequential — no "
                "teacher-forced batching — so this is receiver-side throughput")


def _newest_full_cal():
    """Newest FULL-split calibrated-test measurement recorded by the trainer's
    in-session evaluations (ledger.json). Used to keep the head-to-head honest
    when the loaded report only ran the capped eval (slice noise)."""
    best = None
    pats = [os.path.join(BASE_DIR, "results", "runs", "ledger.json"),
            "/kaggle/input/*/ledger.json", "/kaggle/working/runs/ledger.json"]
    for lp in sorted({p for pat in pats for p in glob.glob(pat)}):
        try:
            with open(lp) as f:
                ledger = json.load(f)
        except Exception:
            continue
        for e in ledger:
            if isinstance(e, dict) and e.get("test_bpb_cal"):
                if best is None or int(e["step"]) > best[0]:
                    best = (int(e["step"]), float(e["test_bpb_cal"]))
    return best


# ---- the suite ---------------------------------------------------------------
def suite_headtohead():
    report, rpath = load_latest_report()
    outdir = hh_outdir(rpath)
    data = hh_test_bytes()
    size_mb = len(data) / 2**20
    hh_dir = os.path.join(outdir, "hh")
    os.makedirs(hh_dir, exist_ok=True)
    inp = os.path.join(hh_dir, "test_split.bin")
    if not (os.path.exists(inp) and os.path.getsize(inp) == len(data)):
        with open(inp, "wb") as f:
            f.write(data)

    print(f"\n[H] head-to-head on the same {size_mb:.1f} MB test split "
          f"(single-threaded child processes) ...")
    rows = []
    for label, spec in _hh_available_codecs():
        r = _hh_run_child(spec, inp)
        if not r.get("ok"):
            print(f"  {label:<38s} SKIP ({r.get('error', 'roundtrip failed')})")
            continue
        if r["comp_s"] < 1.0:                      # re-time fast codecs, median of 3
            reps = [r] + [_hh_run_child(spec, inp) for _ in range(2)]
            r["comp_s"] = statistics.median(x["comp_s"] for x in reps if x.get("ok"))
            r["decomp_s"] = statistics.median(x["decomp_s"] for x in reps if x.get("ok"))
        rows.append(dict(
            name=label, kind="classical", lossless=True,
            bpb=r["out_bytes"] * 8.0 / len(data),
            comp_mb_s=size_mb / r["comp_s"], decomp_mb_s=size_mb / r["decomp_s"],
            core_s_per_100mb=r["comp_s"] / size_mb * 100.0,
            peak_rss_mb=r["maxrss_mb"]))
        print(f"  {label:<38s} {rows[-1]['bpb']:6.4f} bpb   "
              f"comp {rows[-1]['comp_mb_s']:8.2f} MB/s   "
              f"dec {rows[-1]['decomp_mb_s']:8.2f} MB/s   "
              f"rss {rows[-1]['peak_rss_mb']:7.0f} MB")

    # ---- the model's row, from the report (+ live streaming-decode timing) --
    comp, perf, env = report["compression"], report["performance"], report["environment"]
    stream = None
    if MEASURE_STREAM_DECODE:
        try:
            print("  [H] timing true streaming decode (token-by-token) ...")
            stream = measure_streaming_decode()
            print(f"      {stream['tok_s']:.2f} tok/s ≈ {stream['mb_s']*1024:.3f} KB/s "
                  f"over {stream['tokens']} tokens (window {stream['window']})")
        except Exception as e:
            print(f"  [H] streaming decode not measured ({str(e)[:120]})")
    # Density, honestly: classical codecs above compressed the WHOLE split. If
    # this report only ran the capped eval, prefer the newest FULL-split
    # measurement (from ledger.json) as the headline; keep both on record.
    bpb_capped, coverage = comp["test_bpb_cal"], comp.get("coverage_frac", 1.0)
    full = _newest_full_cal() if coverage < 0.9 else None
    bpb_used = max(bpb_capped, full[1]) if full else bpb_capped
    if full:
        print(f"  model density: using full-split {full[1]:.4f} bpb "
              f"(ledger @ step {full[0]:,}) as headline — the loaded report's "
              f"{bpb_capped:.4f} covers only {coverage:.0%} of the split")
    rows.append(dict(
        name="hybrid neural (ours)", kind="model",
        lossless=bool(report["correctness"]["lossless"]),
        bpb=bpb_used, bpb_capped=bpb_capped, coverage_frac=coverage,
        bpb_full=(full[1] if full else None),
        bpb_full_step=(full[0] if full else None),
        comp_mb_s=perf["throughput_mb_s"],
        decomp_mb_s=(stream["mb_s"] if stream else None),
        core_s_per_100mb=100.0 / perf["throughput_mb_s"],
        peak_rss_mb=perf["peak_rss_gb"] * 1024,
        device=env["device"],
        notes="compress speed = teacher-forced batched bound (sender-side); "
              "decompress = measured production incremental streaming decoder "
              "(KV-cache + GRU state); RSS = benchmark-process peak"))

    # ---- decoder artifact footprint (what a receiver must hold) -------------
    exp = PATHS["export_dir"]
    parts = {}
    for fn in ("gate.onnx", "gru_step.onnx", "xformer_window.onnx",
               "order0.f16", "order1.f16", "tokenizer.json"):
        p = os.path.join(exp, fn)
        if os.path.exists(p):
            parts[fn] = os.path.getsize(p)
    for p in glob.glob(os.path.join(os.path.dirname(exp), "*", "o2_*.npz")) + \
             glob.glob(os.path.join(exp, "o2_*.npz")):
        parts["order2_table (o2_*.npz)"] = os.path.getsize(p)
        break
    if not parts:                                  # fallback: fp16 weights estimate
        parts = {"weights fp16 (est.)": int(perf["params_m"] * 2e6),
                 "order1.f16 (est.)": 2 * report["config"]["vocab_size"] ** 2}
    footprint_mb = sum(parts.values()) / 2**20
    print(f"  decoder footprint: {footprint_mb:.0f} MB  "
          f"({', '.join(f'{k} {v/2**20:.1f}MB' for k, v in sorted(parts.items(), key=lambda kv: -kv[1]))})")

    hh = dict(generated=time.strftime("%Y-%m-%d %H:%M:%S"),
              test_bytes=len(data), device=env["device"], host=env.get("host"),
              report_used=rpath, rows=rows, stream_decode=stream,
              decoder_footprint_mb=footprint_mb, decoder_parts=parts)
    with open(os.path.join(outdir, "head_to_head.json"), "w") as f:
        json.dump(hh, f, indent=2, default=float)
    print(f"  [H] saved {os.path.join(outdir, 'head_to_head.json')}")
    return hh


HH = suite_headtohead()

### Suite I — the dashboard (five charts, one question)

1. **Quality** — bpc per codec, best at top. The blue bar is the claim of
   superiority; its label shows the % of bits saved vs the best classical.
2. **Speed** — compress and decompress throughput on a log axis. This is the
   price tag. The two model markers are the bounds explained above.
3. **Trade-off** — density vs speed on one canvas. No codec reaches the
   model's density; nothing near the model's speed. Both facts are the answer.
4. **Cost** — CPU-seconds per 100 MB and peak working memory (log scales).
5. **Break-even** — the economics: total bytes shipped (archive **+ decoder,
   once per corpus**) vs corpus size. Left of the marker the classical codec
   wins end-to-end; right of it the neural archive is smaller *in total*.


In [ ]:
# =============================================================================
#  SECTION 9 -- SUITE I: THE COMPARISON DASHBOARD (quality / speed / size / cost)
# =============================================================================
#  Four charts + one economics chart, all from Suite H's measurements. Design
#  rules: our method is always the single blue emphasis series, classical
#  codecs are the gray context; log scales wherever values span orders of
#  magnitude; every mark is direct-labeled so nothing relies on color alone.
# =============================================================================
PAL.setdefault("de", PAL["axis"])          # de-emphasis gray for context marks
PAL.setdefault("crit", "#d03b3b")          # status: critical (reserved)
PAL.setdefault("warnc", "#fab219")         # status: warning  (reserved)

def _hh_rows(hh):
    rows = sorted(hh["rows"], key=lambda r: r["bpb"])
    ours = next(r for r in rows if r["kind"] == "model")
    return rows, ours


def _show(path):
    try:
        from IPython.display import Image, display
    except ImportError:
        return                                    # plain-python run: PNGs on disk
    if os.path.exists(path):
        display(Image(filename=path))


def chart_hh_bpc(hh, path):
    """Quality: bits/char by codec — ours in blue, classical in gray."""
    rows, ours = _hh_rows(hh)
    fig, ax = _new_ax((7.6, 0.56 * len(rows) + 1.3))
    y = np.arange(len(rows))[::-1]
    best_cl = min(r["bpb"] for r in rows if r["kind"] == "classical")
    for yi, r in zip(y, rows):
        c = PAL["s1"] if r["kind"] == "model" else PAL["de"]
        ax.barh(yi, r["bpb"], height=0.54, color=c, zorder=3)
        label = f"{r['bpb']:.4f}"
        if r["kind"] == "model":
            label += f"   −{(1 - r['bpb'] / best_cl):.0%} vs best classical"
            if r.get("bpb_full"):
                label += f"  (capped slice: {r['bpb_capped']:.4f})"
        ax.text(r["bpb"] + 0.035, yi, label, va="center", fontsize=8.5,
                color=PAL["ink"] if r["kind"] == "model" else PAL["ink2"])
    ax.set_yticks(y, [r["name"] for r in rows], fontsize=9, color=PAL["ink"])
    ax.set_xlim(0, max(r["bpb"] for r in rows) * 1.32)
    ax.set_xlabel("bits per character on the enwik8 test split — lower is better",
                  fontsize=8.5, color=PAL["muted"])
    ax.xaxis.grid(True, color=PAL["grid"], linewidth=0.8, zorder=0)
    _finish(fig, ax, "Quality: the neural compressor needs the fewest bits",
            "same 5 MB held-out bytes for every row; raw text = 8 bits/char; "
            "all rows are byte-exact lossless", path)


def chart_hh_speed(hh, path):
    """Speed: compress + decompress MB/s per codec, log scale (dot pairs)."""
    rows, ours = _hh_rows(hh)
    rows = sorted(rows, key=lambda r: r["comp_mb_s"])
    fig, ax = _new_ax((7.6, 0.62 * len(rows) + 1.5))
    y = np.arange(len(rows))[::-1]
    for yi, r in zip(y, rows):
        xs = [v for v in (r["comp_mb_s"], r["decomp_mb_s"]) if v]
        if len(xs) == 2:
            ax.plot(xs, [yi, yi], color=PAL["grid"], linewidth=1.2, zorder=2)
        ax.scatter([r["comp_mb_s"]], [yi], s=72, color=PAL["s1"], zorder=4,
                   edgecolors=PAL["surface"], linewidths=2)
        if r["decomp_mb_s"]:
            ax.scatter([r["decomp_mb_s"]], [yi], s=72, color=PAL["s2"], zorder=4,
                       edgecolors=PAL["surface"], linewidths=2)
    ax.set_xscale("log")
    labels = [r["name"] + (" *" if r["kind"] == "model" else "") for r in rows]
    ax.set_yticks(y, labels, fontsize=9, color=PAL["ink"])
    ax.set_xlabel("throughput in MB/s — higher is better (log scale)",
                  fontsize=8.5, color=PAL["muted"])
    ax.xaxis.grid(True, color=PAL["grid"], linewidth=0.8, zorder=0)
    hnd = [ax.scatter([], [], s=72, color=PAL["s1"], label="compress"),
           ax.scatter([], [], s=72, color=PAL["s2"], label="decompress")]
    ax.legend(handles=hnd, loc="center left", bbox_to_anchor=(0.03, 0.42),
              frameon=False, fontsize=8.5, labelcolor=PAL["ink2"])
    _finish(fig, ax, "Speed: classical codecs are orders of magnitude faster",
            "* ours: compress = batched teacher-forced bound (sender only); "
            "decompress = measured token-by-token reference decoder", path)


def chart_hh_pareto(hh, path):
    """The trade-off in one picture: density (y) vs compress speed (x, log)."""
    rows, ours = _hh_rows(hh)
    fig, ax = _new_ax((7.6, 4.6))
    ys = [r["bpb"] for r in rows]
    ax.set_xscale("log")
    ax.set_xlim(min(r["comp_mb_s"] for r in rows) * 0.25,
                max(r["comp_mb_s"] for r in rows) * 4.0)
    ax.set_ylim(min(ys) - 0.16, max(ys) + 0.18)
    # greedy label dodging in (log-x, y) space; other DOTS are obstacles too
    dots = [(math.log10(r["comp_mb_s"]), r["bpb"]) for r in rows]
    labels = []

    def _label(i, prefer):
        x, b = dots[i]
        for dx, dy, ha in prefer:
            cx, cy = x + dx / 52.0, b + dy / 110.0
            clear_d = all(abs(cx - px) > 0.16 or abs(cy - py) > 0.055
                          for j, (px, py) in enumerate(dots) if j != i)
            clear_l = all(abs(cx - px) > 0.30 or abs(cy - py) > 0.09
                          for px, py in labels)
            if clear_d and clear_l:
                labels.append((cx, cy))
                return dx, dy, ha
        labels.append((x, b + 0.09))
        return prefer[0]

    order = sorted(range(len(rows)), key=lambda i: rows[i]["bpb"])
    for i in order:
        r = rows[i]
        is_m = r["kind"] == "model"
        c = PAL["s1"] if is_m else PAL["de"]
        ax.scatter([r["comp_mb_s"]], [r["bpb"]], s=110 if is_m else 84,
                   color=c, zorder=4, edgecolors=PAL["surface"], linewidths=2)
        prefer = ([(9, 9, "left")] if is_m else
                  [(0, 10, "center"), (0, -18, "center"),
                   (11, -3, "left"), (-11, -3, "right")])
        dx, dy, ha = _label(i, prefer)
        ax.annotate(r["name"].split(" (")[0], (r["comp_mb_s"], r["bpb"]),
                    xytext=(dx, dy), textcoords="offset points", ha=ha,
                    fontsize=8, color=PAL["ink"] if is_m else PAL["ink2"])
    ax.set_xlabel("compress throughput, MB/s (log) — right is faster",
                  fontsize=8.5, color=PAL["muted"])
    ax.set_ylabel("bits per character — down is denser", fontsize=8.5,
                  color=PAL["muted"])
    ax.yaxis.grid(True, color=PAL["grid"], linewidth=0.8, zorder=0)
    _finish(fig, ax, "The trade-off: nothing matches ours on density — "
            "everything beats it on speed",
            "an ideal codec would sit in the bottom-right corner; the two "
            "'best at something' points are ours (density) and gzip (speed)",
            path)


def chart_hh_cost(hh, path):
    """Computational cost: CPU-core-seconds per 100 MB and peak RSS (log)."""
    import matplotlib.pyplot as plt
    rows, ours = _hh_rows(hh)
    rows = sorted(rows, key=lambda r: r["core_s_per_100mb"])
    fig, axes = plt.subplots(1, 2, figsize=(9.4, 0.6 * len(rows) + 1.6), dpi=150)
    fig.patch.set_facecolor(PAL["surface"])
    y = np.arange(len(rows))[::-1]
    specs = [("core_s_per_100mb", "compress cost — device-seconds per 100 MB"),
             ("peak_rss_mb", "peak working memory — MB")]
    for ax, (key, xlabel) in zip(axes, specs):
        ax.set_facecolor(PAL["surface"])
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color(PAL["axis"])
        ax.tick_params(colors=PAL["muted"], labelsize=8.5, length=0)
        for yi, r in zip(y, rows):
            c = PAL["s1"] if r["kind"] == "model" else PAL["de"]
            ax.scatter([r[key]], [yi], s=84, color=c, zorder=4,
                       edgecolors=PAL["surface"], linewidths=2)
            txt = f"{r[key]:,.0f}" if r[key] >= 10 else f"{r[key]:.2f}"
            ax.annotate(txt, (r[key], yi), xytext=(7, 0), textcoords="offset points",
                        va="center", fontsize=8, color=PAL["ink2"])
        ax.set_xscale("log")
        ax.set_xlabel(xlabel + " (log)", fontsize=8.5, color=PAL["muted"])
        ax.xaxis.grid(True, color=PAL["grid"], linewidth=0.8, zorder=0)
        ax.set_yticks(y, [r["name"].split(" (")[0] for r in rows] if ax is axes[0]
                      else [""] * len(rows), fontsize=9, color=PAL["ink"])
    axes[0].set_title("Computational cost per codec (this machine, single core; "
                      "ours as measured on " + str(hh.get("device")) + ")",
                      loc="left", fontsize=11.5, fontweight="semibold",
                      color=PAL["ink"], pad=14)
    fig.tight_layout()
    fig.savefig(path, facecolor=PAL["surface"], bbox_inches="tight")
    plt.close(fig)
    print(f"  [chart] {path}")


def chart_breakeven(hh, path):
    """Storage economics: total bytes shipped (archive + decoder) vs corpus size."""
    rows, ours = _hh_rows(hh)
    cl = [r for r in rows if r["kind"] == "classical"]
    best = min(cl, key=lambda r: r["bpb"])
    gz = next((r for r in cl if r["name"].startswith("gzip-9")), cl[0])
    fp = hh["decoder_footprint_mb"]
    xs = np.logspace(np.log2(0.05), np.log2(200), 220, base=2)  # GB
    series = [(f"ours + {fp:.0f} MB decoder", ours["bpb"], fp, PAL["s1"]),
              (best["name"].split(" (")[0], best["bpb"], 0.0, PAL["s2"]),
              (gz["name"].split(" (")[0], gz["bpb"], 0.0, PAL["s3"])]
    fig, ax = _new_ax((7.6, 4.6))
    for name, bpb, extra, color in series:
        ship = xs * 1024 * bpb / 8 + extra
        ax.plot(xs, ship, color=color, linewidth=2, zorder=3,
                solid_joinstyle="round")
        ax.annotate(name, (xs[-1], ship[-1]), xytext=(6, 0),
                    textcoords="offset points", va="center", fontsize=8.5,
                    color=PAL["ink2"])
    for name, bpb, extra, color in series[1:]:
        if bpb > ours["bpb"]:
            x_be = fp * 8 / (bpb - ours["bpb"]) / 1024          # GB
            y_be = x_be * 1024 * bpb / 8
            ax.scatter([x_be], [y_be], s=76, color=PAL["s1"], zorder=5,
                       edgecolors=PAL["surface"], linewidths=2)
            ax.annotate(f"break-even vs {name}: {x_be:.1f} GB",
                        (x_be, y_be), xytext=(8, -12), textcoords="offset points",
                        fontsize=8.5, color=PAL["ink"])
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(xs[0], xs[-1] * 7)
    ax.set_xlabel("corpus of similar text to archive, GB (log)",
                  fontsize=8.5, color=PAL["muted"])
    ax.set_ylabel("total MB shipped: archive + decoder (log)",
                  fontsize=8.5, color=PAL["muted"])
    ax.yaxis.grid(True, color=PAL["grid"], linewidth=0.8, zorder=0)
    _finish(fig, ax, "When does the density win pay for the model?",
            "the decoder travels once per corpus; left of the marker classical "
            "wins outright, right of it the neural archive is smaller in total",
            path)


def suite_dashboard(hh):
    print("\n[I] comparison dashboard ...")
    rows, ours = _hh_rows(hh)
    hdr = (f"  {'codec':<38s} {'bpc':>7s} {'ratio':>6s} {'comp MB/s':>10s} "
           f"{'dec MB/s':>10s} {'core-s/100MB':>13s} {'peak RSS':>9s}")
    print(hdr + "\n  " + "-" * (len(hdr) - 2))
    for r in rows:
        dec = f"{r['decomp_mb_s']:10.3g}" if r["decomp_mb_s"] else "         —"
        print(f"  {r['name']:<38s} {r['bpb']:7.4f} {8.0 / r['bpb']:5.2f}x "
              f"{r['comp_mb_s']:10.3g} {dec} {r['core_s_per_100mb']:13,.0f} "
              f"{r['peak_rss_mb']:8,.0f}M")
    outdir = os.path.dirname(hh["report_used"])
    cdir = os.path.join(writable_dir(outdir), "charts")
    os.makedirs(cdir, exist_ok=True)
    charts = {"hh_bpc": chart_hh_bpc, "hh_speed": chart_hh_speed,
              "hh_pareto": chart_hh_pareto, "hh_cost": chart_hh_cost,
              "breakeven": chart_breakeven}
    for name, fn in charts.items():
        p = os.path.join(cdir, f"{name}.png")
        try:
            fn(hh, p)
            _show(p)
        except Exception as e:
            print(f"  [chart] {name} failed (non-fatal): {e}")


suite_dashboard(HH)


### Suite J — goal audit: is 1.245 bpc still the optimal target?

The 1.245 goal was an extrapolation from **two early points**. This suite
redoes it with everything we now know:

1. **Harvest** every `(step, val bpc)` pair from every checkpoint in
   `results/` and any attached Kaggle dataset (cached — re-runs are instant),
   plus every *full* calibrated-test measurement recorded in `ledger.json`.
   Capped-eval numbers are deliberately excluded from the fit (slice noise).
2. **Fit an ensemble** of decay laws to the full trajectory: power laws
   `b∞ + A·s^−α`, exponential saturation, the trainer's linear-rate-decay
   heuristic, and a *no-further-gain* floor.
3. **Convert** val → calibrated-test using the offset **measured** by the
   full in-session evaluations (≈0.02 bpb), not assumed.
4. **Compare** the projection band at curriculum end (step 160k) with 1.245
   and say plainly: `KEEP` / `AT RISK` / `OUT OF REACH → revise to X` /
   `TOO SOFT → tighten`, with the projected value and the plateau evidence
   (how many steps since validation last improved).

The verdict is computed from data at run time — if future sessions improve
the model, re-running this cell updates the answer.


In [ ]:
# =============================================================================
#  SECTION 10 -- SUITE J: GOAL AUDIT — is 1.245 bpc still the right target?
# =============================================================================
#  The 1.245 goal was set on 2026-07-06 from an ensemble of decay models fitted
#  to the first two checkpoints on record (val 1.3100@67k -> 1.3071@72k). Much
#  more evidence exists now. This suite re-runs that analysis properly:
#    1. harvest EVERY (step, val bpb) pair from every checkpoint / ledger it
#       can find (results/, /kaggle/input/*) — cached, so re-runs are instant;
#    2. fit a fresh ensemble (power laws, exponential saturation, linear rate
#       decay, and a no-further-gain floor) to the full trajectory;
#    3. convert to calibrated TEST bpc using the offset measured by the full
#       in-session evaluations recorded in ledger.json;
#    4. compare the projection band at curriculum end (step 160k) against the
#       goal and say plainly: KEEP the goal, or REVISE it (and to what).
# =============================================================================
GOAL_MARGIN_BPB = 0.004      # projection closer than this to the goal = "in play"

def _extract_ckpt_point(path):
    try:
        ck = _xla_safe_load(path, map_location="cpu", weights_only=True)
    except Exception:
        try:
            ck = _xla_safe_load(path, map_location="cpu", weights_only=False)
        except Exception:
            return None
    step = int(ck.get("step", -1))
    val = ck.get("val_bpb", None)
    if val is None:
        val = ck.get("best", None)
    del ck
    if step > 0 and val is not None and math.isfinite(float(val)):
        return step, float(val)
    return None


def harvest_trajectory(outdir):
    """(step, val bpb) from all checkpoints + (step, cal-test bpb, hours) from
    all ledgers. Checkpoint reads are cached by (mtime, size)."""
    cpath = os.path.join(outdir, "trajectory_cache.json")
    cache = {}
    if os.path.exists(cpath):
        with open(cpath) as f:
            cache = json.load(f)

    pats = [os.path.join(BASE_DIR, "results", "runs", "*", "*.pt"),
            os.path.join(BASE_DIR, "results", "runs", "*", "*.last"),
            os.path.join(BASE_DIR, "results", "export", "*.pt"),
            os.path.join(BASE_DIR, "results", "export", "*.last"),
            "/kaggle/input/*/v*/*.pt", "/kaggle/input/*/v*/*.last",
            "/kaggle/input/*/export/*.pt", "/kaggle/input/*/export/*.last",
            "/kaggle/working/runs/*/*.pt", "/kaggle/working/runs/*/*.last"]
    files = sorted({p for pat in pats for p in glob.glob(pat)})
    val_pts, fresh = {}, 0
    for p in files:
        st = os.stat(p)
        key, sig = p, f"{st.st_mtime_ns}:{st.st_size}"
        hit = cache.get(key)
        if hit and hit.get("sig") == sig:
            pt = hit.get("pt")
        else:
            pt = _extract_ckpt_point(p)
            cache[key] = dict(sig=sig, pt=pt)
            fresh += 1
        if pt:
            s, v = pt
            val_pts[s] = min(v, val_pts.get(s, float("inf")))
    with open(cpath, "w") as f:
        json.dump(cache, f)
    print(f"  [J] {len(files)} checkpoints scanned ({fresh} newly read, rest "
          f"cached) -> {len(val_pts)} distinct (step, val bpb) points")

    cal_pts, offsets = [], []
    led_pats = [os.path.join(BASE_DIR, "results", "runs", "ledger.json"),
                "/kaggle/input/*/ledger.json", "/kaggle/working/runs/ledger.json"]
    for lp in sorted({p for pat in led_pats for p in glob.glob(pat)}):
        try:
            with open(lp) as f:
                ledger = json.load(f)
        except Exception:
            continue
        for e in ledger:
            if isinstance(e, dict) and e.get("test_bpb_cal"):
                cal_pts.append((int(e["step"]), float(e["test_bpb_cal"]),
                                float(e.get("elapsed_h", float("nan")))))
                near = min(val_pts, key=lambda s: abs(s - e["step"]), default=None)
                if near is not None and abs(near - e["step"]) <= 3000:
                    offsets.append(val_pts[near] - e["test_bpb_cal"])
    cal_pts = sorted(set(cal_pts))
    offset = float(np.median(offsets)) if offsets else 0.0378  # accumulated.json fallback
    print(f"  [J] {len(cal_pts)} full calibrated-test measurements in ledgers; "
          f"val->test offset = {offset:+.4f} bpb "
          f"({'measured' if offsets else 'fallback'})")
    return sorted(val_pts.items()), cal_pts, offset


def _lsq_two(x, y):
    """least squares for y = c0 + c1*x; returns (c0, c1, rmse)."""
    A = np.stack([np.ones_like(x), x], 1)
    (c0, c1), *_ = np.linalg.lstsq(A, y, rcond=None)
    r = float(np.sqrt(np.mean((c0 + c1 * x - y) ** 2)))
    return float(c0), float(c1), r


def fit_ensemble(val_pts, total_steps):
    """Fit decay families to (step, val); return curve fns + projections."""
    s = np.array([p[0] for p in val_pts], dtype=float)
    b = np.array([p[1] for p in val_pts], dtype=float)
    members = []
    for alpha in np.arange(0.15, 2.51, 0.05):             # power: b_inf + A*s^-a
        c0, c1, r = _lsq_two(s ** -alpha, b)
        if c1 > 0 and 0 <= c0 < b.min() + 0.05:
            members.append(dict(name=f"power a={alpha:.2f}", rmse=r,
                                fn=(lambda c0, c1, a: lambda q: c0 + c1 * q ** -a)(c0, c1, alpha)))
    for tau in np.logspace(np.log10(4e3), np.log10(4e5), 28):   # exp saturation
        c0, c1, r = _lsq_two(np.exp(-s / tau), b)
        if c1 > 0 and 0 <= c0 < b.min() + 0.05:
            members.append(dict(name=f"exp tau={tau/1e3:.0f}k", rmse=r,
                                fn=(lambda c0, c1, t: lambda q: c0 + c1 * np.exp(-q / t))(c0, c1, tau)))
    if len(val_pts) >= 2:                                  # trainer's own heuristic
        (s1, b1), (s2, b2) = val_pts[0], val_pts[-1]
        proj = estimate_final_bpb(s1, b1, s2, b2, total_steps)
        if proj is not None:
            rate = (b1 - b2) / max(1, s2 - s1)
            r0 = rate / max(1e-9, 1 - 0.5 * (s1 + s2) / total_steps)
            members.append(dict(
                name="linear rate decay", rmse=float("nan"),
                fn=(lambda b2, s2, r0, T: lambda q: b2 - r0 * (
                    np.clip(q - s2, 0, None)
                    - (np.clip(q, s2, None) ** 2 - s2 ** 2) / (2 * T)))(b2, s2, r0, total_steps)))
    members.append(dict(name="no further gain (floor)", rmse=float("nan"),
                        fn=(lambda m: lambda q: np.full_like(np.asarray(q, float), m))(b.min())))
    # keep well-fitting members: rmse <= 2x best fitted rmse (heuristics always kept)
    fitted = [m for m in members if math.isfinite(m["rmse"])]
    if fitted:
        best = min(m["rmse"] for m in fitted)
        members = [m for m in members
                   if not math.isfinite(m["rmse"]) or m["rmse"] <= max(2 * best, 1e-4)]
    for m in members:
        m["proj"] = float(m["fn"](total_steps))
    return members


def chart_goal_audit(val_pts, cal_pts, members, offset, goal, total, verdict,
                     reachable, path):
    fig, ax = _new_ax((7.8, 4.6))
    grid = np.linspace(min(p[0] for p in val_pts), total, 240)
    curves = np.stack([np.asarray(m["fn"](grid), dtype=float) - offset
                       for m in members])
    ax.fill_between(grid, curves.min(0), curves.max(0), color=PAL["s1"],
                    alpha=0.10, zorder=1, linewidth=0)
    for c in curves[:: max(1, len(curves) // 16)]:      # a sample, not a smear
        ax.plot(grid, c, color=PAL["s1"], alpha=0.16, linewidth=0.9, zorder=2)
    med = np.median(curves, 0)
    ax.plot(grid, med, color=PAL["s1"], linewidth=2, linestyle=(0, (4, 3)),
            zorder=3)
    ax.annotate(f"ensemble median ≈ {med[-1]:.4f}", (grid[-1], med[-1]),
                xytext=(-6, -20), textcoords="offset points", ha="right",
                fontsize=8.5, color=PAL["ink2"])
    xs = [p[0] for p in val_pts]
    ys = [p[1] - offset for p in val_pts]
    ax.plot(xs, ys, color=PAL["s1"], linewidth=2, zorder=4)
    ax.scatter(xs, ys, s=56, color=PAL["s1"], zorder=5,
               edgecolors=PAL["surface"], linewidths=2, label="val checkpoints (shifted by offset)")
    if cal_pts:
        ax.scatter([p[0] for p in cal_pts], [p[1] for p in cal_pts], s=72,
                   color=PAL["s2"], zorder=6, edgecolors=PAL["surface"],
                   linewidths=2, label="full calibrated-test measurements")
    gcol = PAL["good"] if reachable else PAL["crit"]
    ax.axhline(goal, color=gcol, linewidth=1.4, zorder=2)
    ax.text(0.01, goal, f" goal {goal} — {verdict}",
            transform=ax.get_yaxis_transform(), fontsize=8.5,
            color="#006300" if reachable else PAL["crit"], va="bottom")
    ax.set_xlabel("optimizer step", fontsize=8.5, color=PAL["muted"])
    ax.set_ylabel("calibrated test bits/char", fontsize=8.5, color=PAL["muted"])
    ax.yaxis.grid(True, color=PAL["grid"], linewidth=0.8, zorder=0)
    ax.xaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
    ax.legend(loc="upper right", frameon=False, fontsize=8.5, labelcolor=PAL["ink2"])
    _finish(fig, ax, "Goal audit: every decay model, fitted to the full history",
            "solid blue: observed; thin lines + band: fitted ensemble to "
            "curriculum end; dashed: ensemble median", path)


def suite_goal_audit(goal_bpb=None):
    print("\n[J] goal audit ...")
    goal = goal_bpb if goal_bpb is not None else GOAL_TEST_BPB
    report, rpath = load_latest_report()
    outdir = hh_outdir(rpath)
    cfg_tot = (report["config"]["stage1_steps"] + report["config"]["stage2_steps"])
    val_pts, cal_pts, offset = harvest_trajectory(outdir)
    if len(val_pts) < 3:
        print("  [J] not enough trajectory points for a fit — need >= 3")
        return None
    members = fit_ensemble(val_pts, cfg_tot)
    projs_t = sorted(m["proj"] - offset for m in members)
    lo_t, med_t, hi_t = projs_t[0], float(np.median(projs_t)), projs_t[-1]

    # plateau evidence: how long has best-val been stuck?
    best_val = min(v for _, v in val_pts)
    first_best = min(s for s, v in val_pts if v <= best_val + 1e-9)
    stalled = max(s for s, _ in val_pts) - first_best

    # marginal rate from the ledgers' full measurements (bpb per GPU-hour)
    rate_h = None
    tl = [(h, c) for _, c, h in cal_pts if math.isfinite(h)]
    if len(tl) >= 2:
        (h1, c1), (h2, c2) = tl[0], tl[-1]
        if h2 > h1:
            rate_h = (c1 - c2) / (h2 - h1)

    goal_met = bool(report["stopping"].get("goal_met"))
    reachable = lo_t <= goal + GOAL_MARGIN_BPB
    if goal_met:
        verdict, why = "GOAL MET", "the exported model already reached it."
    elif hi_t <= goal - GOAL_MARGIN_BPB:
        verdict, why = "GOAL TOO SOFT — TIGHTEN", (
            f"even the worst ensemble member lands at {hi_t:.4f}; "
            f"a goal of {med_t:.3f} would still be safe.")
    elif reachable and med_t <= goal + GOAL_MARGIN_BPB:
        verdict, why = "KEEP 1.245-CLASS GOAL", (
            f"the projection band [{lo_t:.4f}, {hi_t:.4f}] brackets the goal; "
            f"curriculum end decides.")
    elif reachable:
        verdict, why = "AT RISK", (
            f"only the most optimistic members reach it "
            f"(band [{lo_t:.4f}, {hi_t:.4f}], median {med_t:.4f}).")
    else:
        verdict, why = "OUT OF REACH FOR THIS TIER — REVISE", (
            f"even the best-case member lands at {lo_t:.4f} > goal {goal}. "
            f"The evidence-based target for THIS architecture is "
            f"≈{med_t:.3f} bpc; 1.245 remains valid only as the target for "
            f"the next ladder tier (d_model 384, 5 layers).")

    print(f"  fitted {len(members)} ensemble members on {len(val_pts)} points")
    print(f"  projection at step {cfg_tot // 1000}k (calibrated test bpc): "
          f"median {med_t:.4f}   band [{lo_t:.4f}, {hi_t:.4f}]   goal {goal}")
    print(f"  best val bpb {best_val:.4f} first reached at step {first_best:,} "
          f"-> no improvement for {stalled:,} steps")
    if rate_h is not None:
        need = (cal_pts[-1][1] - goal)
        eta = need / rate_h if rate_h > 0 else float("inf")
        print(f"  measured full-eval rate {rate_h:+.5f} bpb/h -> the remaining "
              f"{need:+.4f} bpb would need ≈{eta:,.0f} more GPU-hours at that rate")
    print(f"  VERDICT: {verdict} — {why}")

    cdir = os.path.join(writable_dir(outdir), "charts")
    os.makedirs(cdir, exist_ok=True)
    cpath = os.path.join(cdir, "goal_audit.png")
    try:
        chart_goal_audit(val_pts, cal_pts, members, offset, goal,
                         cfg_tot, verdict, reachable, cpath)
        _show(cpath)
    except Exception as e:
        print(f"  [chart] goal_audit failed (non-fatal): {e}")

    audit = dict(generated=time.strftime("%Y-%m-%d %H:%M:%S"), goal=goal,
                 verdict=verdict, reason=why, projection_median=med_t,
                 projection_band=[lo_t, hi_t], offset_val_to_test=offset,
                 n_val_points=len(val_pts), n_members=len(members),
                 val_points=val_pts, cal_points=cal_pts,
                 best_val_bpb=best_val, stalled_steps=int(stalled),
                 rate_bpb_per_hour=rate_h,
                 members=[dict(name=m["name"],
                               rmse=None if math.isnan(m.get("rmse", float("nan"))) else m["rmse"],
                               proj_val=m["proj"], proj_test=m["proj"] - offset)
                          for m in members])
    with open(os.path.join(outdir, "goal_audit.json"), "w") as f:
        json.dump(audit, f, indent=2, default=float)
    print(f"  [J] saved {os.path.join(outdir, 'goal_audit.json')}")
    return audit


GOAL_AUDIT = suite_goal_audit()


### Suite K — the answer

Everything condensed: a KPI row (density win, speed price, break-even corpus,
goal verdict) and a numbered plain-language answer, saved as `verdict.md`.
Every figure in it is pulled from the measured artifacts
(`benchmark_report.json`, `head_to_head.json`, `goal_audit.json`) — nothing is
typed in by hand, so the cell stays true after retraining or re-measurement.


In [ ]:
# =============================================================================
#  SECTION 11 -- THE ANSWER (one KPI row + one paragraph + verdict.md)
# =============================================================================
#  Everything above condensed into the reply to the single question. All
#  numbers are pulled from the measured artifacts (benchmark_report.json,
#  head_to_head.json, goal_audit.json) — nothing here is typed in by hand.
# =============================================================================
def _fmt_x(v):
    return f"{v:,.0f}×" if v >= 10 else f"{v:.1f}×"


def build_answer(hh, audit):
    report, _ = load_latest_report()
    rows, ours = _hh_rows(hh)
    cl = [r for r in rows if r["kind"] == "classical"]
    best = min(cl, key=lambda r: r["bpb"])
    fast = max(cl, key=lambda r: r["comp_mb_s"])
    gz = next((r for r in cl if r["name"].startswith("gzip-9")), cl[0])
    fp = hh["decoder_footprint_mb"]
    be_gb = fp * 8 / (best["bpb"] - ours["bpb"]) / 1024 if best["bpb"] > ours["bpb"] else float("inf")

    a = dict(
        quality_pct=(1 - ours["bpb"] / best["bpb"]) * 100,
        quality_x=best["bpb"] / ours["bpb"],
        vs_gzip_x=gz["bpb"] / ours["bpb"],
        saved_mb_per_100mb=(best["bpb"] - ours["bpb"]) / 8 * 100,
        best_name=best["name"].split(" (")[0],
        fast_name=fast["name"].split(" (")[0],
        enc_slow_x=best["comp_mb_s"] / max(1e-12, ours["comp_mb_s"]),
        enc_slow_fast_x=fast["comp_mb_s"] / max(1e-12, ours["comp_mb_s"]),
        dec_slow_x=(best["decomp_mb_s"] / ours["decomp_mb_s"]
                    if ours.get("decomp_mb_s") else None),
        rss_x=ours["peak_rss_mb"] / best["peak_rss_mb"],
        cost_x=ours["core_s_per_100mb"] / best["core_s_per_100mb"],
        footprint_mb=fp, breakeven_gb=be_gb,
        ours_bpc=ours["bpb"], best_bpc=best["bpb"],
        ours_bpc_capped=ours.get("bpb_capped"),
        ours_full_step=ours.get("bpb_full_step"),
        lossless=ours["lossless"], device=hh.get("device"),
        goal=audit["goal"] if audit else GOAL_TEST_BPB,
        goal_verdict=audit["verdict"] if audit else "n/a",
        goal_reason=audit["reason"] if audit else "",
        goal_proj=audit["projection_median"] if audit else None,
        goal_band=audit["projection_band"] if audit else None,
        gates_passed=bool(report.get("gates_passed")))
    return a


def chart_verdict_tiles(a, path):
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    keep = a["goal_verdict"].startswith(("KEEP", "GOAL MET", "GOAL TOO SOFT"))
    goal_short = {"KEEP": "Keep", "GOAL MET": "Met ✓", "AT RISK": "At risk",
                  "GOAL TOO SOFT": "Tighten", "OUT OF REACH":
                  (f"Revise → {a['goal_proj']:.3f}" if a["goal_proj"] else "Revise")}
    gval = next((v for k, v in goal_short.items()
                 if a["goal_verdict"].startswith(k)), a["goal_verdict"][:14])
    tiles = [
        ("Density vs best classical", f"−{a['quality_pct']:.0f}%",
         f"{a['ours_bpc']:.3f} vs {a['best_bpc']:.3f} bpc ({a['best_name']})",
         "#006300"),
        ("Compress speed vs " + a["best_name"], _fmt_x(a["enc_slow_x"]) + " slower",
         f"and {_fmt_x(a['enc_slow_fast_x'])} slower than {a['fast_name']}",
         PAL["crit"]),
        ("Break-even corpus", ("∞" if not math.isfinite(a["breakeven_gb"])
                               else f"{a['breakeven_gb']:.1f} GB"),
         f"decoder travels once: {a['footprint_mb']:.0f} MB",
         PAL["ink2"]),
        (f"Goal {a['goal']} bpc", gval,
         (f"projection {a['goal_proj']:.3f} "
          f"[{a['goal_band'][0]:.3f}–{a['goal_band'][1]:.3f}]"
          if a["goal_proj"] else ""),
         "#006300" if keep else PAL["crit"]),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(11.6, 2.1), dpi=150)
    fig.patch.set_facecolor(PAL["surface"])
    for ax, (label, value, sub, col) in zip(axes, tiles):
        ax.set_axis_off()
        ax.add_patch(FancyBboxPatch((0.02, 0.04), 0.96, 0.92,
                                    boxstyle="round,pad=0.02,rounding_size=0.04",
                                    facecolor=PAL["surface"], edgecolor=PAL["grid"],
                                    linewidth=1.0, transform=ax.transAxes))
        ax.text(0.09, 0.76, label, transform=ax.transAxes, fontsize=9,
                color=PAL["muted"], va="center")
        ax.text(0.09, 0.47, value, transform=ax.transAxes,
                fontsize=(16 if len(value) <= 14 else 13),
                fontweight="semibold", color=PAL["ink"], va="center")
        ax.text(0.09, 0.20, sub, transform=ax.transAxes, fontsize=7.8,
                color=col, va="center")
    fig.suptitle("Is it superior to the traditional methods?  Where it wins — "
                 "and what it costs", x=0.005, y=1.04, ha="left", fontsize=12,
                 fontweight="semibold", color=PAL["ink"])
    fig.tight_layout()
    fig.savefig(path, facecolor=PAL["surface"], bbox_inches="tight")
    plt.close(fig)
    print(f"  [chart] {path}")


def the_answer(hh, audit):
    a = build_answer(hh, audit)
    outdir = hh_outdir(hh["report_used"])
    cdir = os.path.join(outdir, "charts")
    os.makedirs(cdir, exist_ok=True)
    tpath = os.path.join(cdir, "verdict_tiles.png")
    try:
        chart_verdict_tiles(a, tpath)
        _show(tpath)
    except Exception as e:
        print(f"  [chart] verdict tiles failed (non-fatal): {e}")

    dec_txt = (f"true streaming decode measured at "
               f"{hh['stream_decode']['mb_s']*1024:.2f} KB/s"
               if hh.get("stream_decode") else "streaming decode not measured here")
    verdict_lines = [
        "# The verdict", "",
        f"*(auto-generated {time.strftime('%Y-%m-%d %H:%M:%S')} from measured "
        f"artifacts on {a['device']}; correctness gates "
        f"{'PASS' if a['gates_passed'] else 'FAIL'})*", "",
        "**Question: well calibrated and implemented, is the neural compressor "
        "superior to the traditional methods — and by how much?**", "",
        f"1. **Quality — yes, clearly.** {a['ours_bpc']:.4f} bits/char"
        + (f" (full split; the capped slice estimates "
           f"{a['ours_bpc_capped']:.4f})" if a["ours_full_step"] else "")
        + f", byte-exact lossless, vs {a['best_bpc']:.4f} for the best "
        f"classical codec tested ({a['best_name']}): "
        f"**{a['quality_pct']:.0f}% fewer bits** "
        f"({a['quality_x']:.2f}× denser; {a['vs_gzip_x']:.2f}× vs gzip-9). "
        f"Every 100 MB of enwik8-like text ships "
        f"{a['saved_mb_per_100mb']:.1f} MB smaller than {a['best_name']}.",
        f"2. **Speed — no, by orders of magnitude.** Compression runs "
        f"{_fmt_x(a['enc_slow_x'])} slower than {a['best_name']} and "
        f"{_fmt_x(a['enc_slow_fast_x'])} slower than {a['fast_name']} "
        f"(and that is the batched sender-side bound; {dec_txt}).",
        f"3. **Size — wins only past break-even.** The receiver must hold a "
        f"{a['footprint_mb']:.0f} MB decoder bundle, so the density win pays "
        f"for itself only for ≥ **{a['breakeven_gb']:.1f} GB** of similar text "
        f"(vs {a['best_name']}); below that, classical wins end-to-end.",
        f"4. **Computational cost — no.** ≈{_fmt_x(a['cost_x'])} the CPU cost "
        f"per byte of {a['best_name']} and ≈{_fmt_x(a['rss_x'])} its working "
        f"memory.",
        f"5. **Goal audit.** {a['goal_verdict']} — {a['goal_reason']}", "",
        "**Bottom line:** as a *density* technology for large, similar-domain "
        "text archives (or bandwidth-priced pipelines), it is measurably "
        "superior; as a general-purpose codec it is not — speed, memory and "
        "footprint still belong to the classical methods.", ""]
    with open(os.path.join(outdir, "verdict.md"), "w") as f:
        f.write("\n".join(verdict_lines))
    print(f"  [K] saved {os.path.join(outdir, 'verdict.md')}\n")
    print("\n".join(verdict_lines[4:]))
    return a


ANSWER = the_answer(HH, GOAL_AUDIT)


---
### Artifacts this notebook writes

| File | Producer | Contents |
|---|---|---|
| `benchmark_report.json` / `.md` | Part 1 RUN CELL | all suite A–G measurements |
| `charts/*.png` | Parts 1 & 2 | every figure shown inline |
| `head_to_head.json` | Suite H | codec-vs-model measurements |
| `goal_audit.json` | Suite J | trajectory, ensemble fits, goal verdict |
| `verdict.md` | Suite K | the final answer, regenerable |

### Caveats to keep the conclusions honest

* All density numbers are **enwik8-domain** (English Wikipedia text). The
  model's advantage is learned domain structure; on unrelated content types
  Suite E shows classical codecs win, sometimes massively.
* Speed/cost numbers are **this machine's** (see the report's `environment`);
  ratios between codecs travel better than absolute MB/s.
* The strongest published context-mixing programs (cmix/NNCP-class) reach
  ≈1 bpc on *full* enwik8 — but under a different protocol (self-contained,
  model learned during the pass, days of compute). Our numbers answer the
  *shared-model archival* question instead; the break-even chart is what makes
  that comparison fair.
* Capped vs full evaluation differ by ≈0.02 bpc on this checkpoint (slice
  noise). Where it matters, the notebook says which one it is using.
